# 🏦 Home Credit Default Risk — Data Engineering Notebook

## 📌 Project Overview

This project is based on the **Home Credit Default Risk** dataset from Kaggle.

The goal of the full project is to build a machine learning model that can predict whether a loan applicant may face repayment difficulty.

However, this notebook focuses only on the **data engineering stage before EDA**.

---

## 🎯 Notebook Objective

The objective of this notebook is to prepare a clean and reliable data foundation before starting exploratory data analysis.

In this notebook, we will:

* Load all raw dataset files
* Check data size and memory usage
* Understand table structure
* Validate important ID columns
* Check train/test consistency
* Identify missing values, duplicates, and anomalies
* Plan safe aggregation and integration
* Save clean validated files for the next notebook

---

## 🔗 Useful Links

* [Kaggle Competition Page](https://www.kaggle.com/competitions/home-credit-default-risk)
* [Kaggle Dataset Description](https://www.kaggle.com/competitions/home-credit-default-risk/data)
* [Home Credit Group](https://www.homecredit.net/)
* [Pandas Documentation](https://pandas.pydata.org/docs/)
* [Scikit-learn Documentation](https://scikit-learn.org/stable/)

---

## 🧠 Main Data Engineering Rule

The final dataset must stay at applicant level.

```text
1 SK_ID_CURR = 1 applicant = 1 row
```

Historical tables such as `bureau`, `bureau_balance`, `previous_application`, `POS_CASH_balance`, `installments_payments`, and `credit_card_balance` contain multiple rows per applicant.

So, these tables must be aggregated before merging with the main application table.

---

## ⚠️ What This Notebook Will Not Do

This notebook will not include:

* Exploratory data analysis
* Target pattern analysis
* Feature selection
* Model training
* Hyperparameter tuning
* Final prediction generation

Those steps will come later.

---

## 📁 Expected Dataset Files

The raw dataset should contain the following files:

```text
application_train.csv
application_test.csv
bureau.csv
bureau_balance.csv
previous_application.csv
POS_CASH_balance.csv
installments_payments.csv
credit_card_balance.csv
HomeCredit_columns_description.csv
sample_submission.csv
```

---

## ✅ Expected Output of This Notebook

By the end of this notebook, we should have:

```text
outputs/data_engineering/
├── data_file_inventory.csv
├── memory_usage_report.csv
├── schema_summary.csv
├── key_validation_report.csv
├── train_test_check_report.csv
├── missingness_report.csv
├── duplicate_report.csv
├── anomaly_report.csv
└── data_engineering_handoff_checklist.csv
```

And validated staged files:

```text
data/interim/validated/
├── application_train_validated.parquet
├── application_test_validated.parquet
├── bureau_validated.parquet
├── bureau_balance_validated.parquet
├── previous_application_validated.parquet
├── POS_CASH_balance_validated.parquet
├── installments_payments_validated.parquet
└── credit_card_balance_validated.parquet
```


## 1.1 Load and Basic Data Check

In this section, we load all raw CSV files into pandas dataframes.

The goal is not to clean or transform the data yet.
The goal is only to confirm that every file loads correctly and to create a simple inventory of the dataset.

We will check:

* File availability
* Row count
* Column count
* Memory usage
* Important ID columns
* Whether `TARGET` exists

No merging, cleaning, encoding, or feature engineering will be done in this section.


In [1]:
from pathlib import Path
import pandas as pd

# Find project root
current_dir = Path.cwd()

if (current_dir / "Data" / "Raw").exists():
    PROJECT_DIR = current_dir
elif (current_dir.parent / "Data" / "Raw").exists():
    PROJECT_DIR = current_dir.parent
else:
    raise FileNotFoundError("Could not find Data/Raw folder. Please check your project structure.")

RAW_DATA_DIR = PROJECT_DIR / "Data" / "Raw"
OUTPUT_DIR = PROJECT_DIR / "Data" / "Processed" / "data_engineering"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", Path(PROJECT_DIR))
print("Raw data directory:", Path(RAW_DATA_DIR))
print("Output directory:", Path(OUTPUT_DIR))

Project directory: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\Credit_Default_Risk
Raw data directory: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\Credit_Default_Risk\Data\Raw
Output directory: c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\Credit_Default_Risk\Data\Processed\data_engineering


In [2]:
# Expected dataset files
file_registry = {
    "application_train": "application_train.csv",
    "application_test": "application_test.csv",
    "bureau": "bureau.csv",
    "bureau_balance": "bureau_balance.csv",
    "previous_application": "previous_application.csv",
    "pos_cash_balance": "POS_CASH_balance.csv",
    "installments_payments": "installments_payments.csv",
    "credit_card_balance": "credit_card_balance.csv",
    "columns_description": "HomeCredit_columns_description.csv",
    "sample_submission": "sample_submission.csv",
}

file_registry

{'application_train': 'application_train.csv',
 'application_test': 'application_test.csv',
 'bureau': 'bureau.csv',
 'bureau_balance': 'bureau_balance.csv',
 'previous_application': 'previous_application.csv',
 'pos_cash_balance': 'POS_CASH_balance.csv',
 'installments_payments': 'installments_payments.csv',
 'credit_card_balance': 'credit_card_balance.csv',
 'columns_description': 'HomeCredit_columns_description.csv',
 'sample_submission': 'sample_submission.csv'}

In [3]:
# Load all CSV files with encoding fallback
data = {}

encoding_options = ["utf-8", "utf-8-sig", "cp1252", "latin1"]

for table_name, file_name in file_registry.items():
    file_path = RAW_DATA_DIR / file_name
    
    if not file_path.exists():
        data[table_name] = None
        print(f"{table_name}: file not found | {file_name}")
        continue
    
    loaded = False
    
    for encoding in encoding_options:
        try:
            data[table_name] = pd.read_csv(file_path, encoding=encoding)
            print(f"{table_name}: loaded successfully | shape = {data[table_name].shape} | encoding = {encoding}")
            loaded = True
            break
        except UnicodeDecodeError:
            continue
    
    if not loaded:
        data[table_name] = None
        print(f"{table_name}: failed to load due to encoding issue")

application_train: loaded successfully | shape = (307511, 122) | encoding = utf-8
application_test: loaded successfully | shape = (48744, 121) | encoding = utf-8
bureau: loaded successfully | shape = (1716428, 17) | encoding = utf-8
bureau_balance: loaded successfully | shape = (27299925, 3) | encoding = utf-8
previous_application: loaded successfully | shape = (1670214, 37) | encoding = utf-8
pos_cash_balance: loaded successfully | shape = (10001358, 8) | encoding = utf-8
installments_payments: loaded successfully | shape = (13605401, 8) | encoding = utf-8
credit_card_balance: loaded successfully | shape = (3840312, 23) | encoding = utf-8
columns_description: loaded successfully | shape = (219, 5) | encoding = cp1252
sample_submission: loaded successfully | shape = (48744, 2) | encoding = utf-8


In [4]:
# Create basic file inventory
id_columns = ["SK_ID_CURR", "SK_ID_PREV", "SK_ID_BUREAU"]

inventory_records = []

for table_name, file_name in file_registry.items():
    df = data[table_name]
    file_path = RAW_DATA_DIR / file_name
    
    if df is not None:
        found_ids = [col for col in id_columns if col in df.columns]
        
        inventory_records.append({
            "table_name": table_name,
            "file_name": file_name,
            "file_exists": True,
            "rows": df.shape[0],
            "columns": df.shape[1],
            "memory_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
            "id_columns_found": ", ".join(found_ids) if found_ids else "None",
            "has_TARGET": "TARGET" in df.columns,
            "status": "Loaded"
        })
    else:
        inventory_records.append({
            "table_name": table_name,
            "file_name": file_name,
            "file_exists": False,
            "rows": None,
            "columns": None,
            "memory_mb": None,
            "id_columns_found": "Not loaded",
            "has_TARGET": None,
            "status": "Missing"
        })

data_file_inventory = pd.DataFrame(inventory_records)

data_file_inventory

,table_name,file_name,file_exists,rows,columns,memory_mb,id_columns_found,has_TARGET,status
0,application_train,application_train.csv,True,307511,122,536.69,SK_ID_CURR,True,Loaded
1,application_test,application_test.csv,True,48744,121,84.74,SK_ID_CURR,False,Loaded
2,bureau,bureau.csv,True,1716428,17,512.11,"SK_ID_CURR, SK_ID_BUREAU",False,Loaded
3,bureau_balance,bureau_balance.csv,True,27299925,3,1926.61,SK_ID_BUREAU,False,Loaded
4,previous_application,previous_application.csv,True,1670214,37,1900.63,"SK_ID_CURR, SK_ID_PREV",False,Loaded
5,pos_cash_balance,POS_CASH_balance.csv,True,10001358,8,1137.25,"SK_ID_CURR, SK_ID_PREV",False,Loaded
6,installments_payments,installments_payments.csv,True,13605401,8,830.41,"SK_ID_CURR, SK_ID_PREV",False,Loaded
7,credit_card_balance,credit_card_balance.csv,True,3840312,23,875.69,"SK_ID_CURR, SK_ID_PREV",False,Loaded
8,columns_description,HomeCredit_columns_description.csv,True,219,5,0.08,None,False,Loaded
9,sample_submission,sample_submission.csv,True,48744,2,0.74,SK_ID_CURR,True,Loaded


In [5]:
# Save inventory report inside Data/Processed/data_engineering
inventory_path = OUTPUT_DIR / "data_file_inventory.csv"

data_file_inventory.to_csv(inventory_path, index=False)

print("Inventory report saved at:")
print(inventory_path)

Inventory report saved at:
c:\Users\USER\Kawsar_Ahmmed\ALL_Projects_Lab\Machine_Learning\Credit_Default_Risk\Data\Processed\data_engineering\data_file_inventory.csv


### Step 1.1 Insight Summary

All expected dataset files loaded successfully, so the raw data setup is correct. The main training table has **307,511 rows and 122 columns**, while the test table has **48,744 rows and 121 columns**, which matches the expected structure because `TARGET` exists only in train.

The dataset is clearly heavy and relational: `bureau_balance` has about **27.3 million rows**, `installments_payments` has **13.6 million rows**, and `POS_CASH_balance` has **10 million+ rows**. So later we must use aggregation carefully and should save staged files as Parquet to avoid slow processing.

The ID structure also looks correct: `application` uses `SK_ID_CURR`, `bureau_balance` uses `SK_ID_BUREAU`, and previous/POS/installments/credit-card tables use `SK_ID_CURR` + `SK_ID_PREV`. This confirms our plan that historical tables must be aggregated before merging.

## 1.2 Memory Check and Storage Decision

From the initial file inventory, we already have row count, column count, and memory usage for each table.

In this section, we only decide which files should be staged as Parquet for faster future loading and aggregation.

Decision:
- Main data files will be converted to Parquet.
- Helper files such as `HomeCredit_columns_description.csv` and `sample_submission.csv` will remain as CSV.
- The decision report will be saved as `memory_usage_report.csv`.

In [6]:
# Create memory usage and storage decision report

memory_report = data_file_inventory.copy()

helper_tables = ["columns_description", "sample_submission"]

memory_report["storage_decision"] = memory_report["table_name"].apply(
    lambda x: "Keep CSV" if x in helper_tables else "Convert to Parquet"
)

memory_report["decision_reason"] = memory_report["table_name"].apply(
    lambda x: "Small helper file; CSV is enough."
    if x in helper_tables
    else "Main dataset file; will be reused in later notebooks."
)

memory_report = memory_report[
    [
        "table_name",
        "file_name",
        "rows",
        "columns",
        "memory_mb",
        "storage_decision",
        "decision_reason",
    ]
]

memory_report_path = OUTPUT_DIR / "memory_usage_report.csv"
memory_report.to_csv(memory_report_path, index=False)

memory_report

,table_name,file_name,rows,columns,memory_mb,storage_decision,decision_reason
0,application_train,application_train.csv,307511,122,536.69,Convert to Parquet,Main dataset file; will be reused in later not...
1,application_test,application_test.csv,48744,121,84.74,Convert to Parquet,Main dataset file; will be reused in later not...
2,bureau,bureau.csv,1716428,17,512.11,Convert to Parquet,Main dataset file; will be reused in later not...
3,bureau_balance,bureau_balance.csv,27299925,3,1926.61,Convert to Parquet,Main dataset file; will be reused in later not...
4,previous_application,previous_application.csv,1670214,37,1900.63,Convert to Parquet,Main dataset file; will be reused in later not...
5,pos_cash_balance,POS_CASH_balance.csv,10001358,8,1137.25,Convert to Parquet,Main dataset file; will be reused in later not...
6,installments_payments,installments_payments.csv,13605401,8,830.41,Convert to Parquet,Main dataset file; will be reused in later not...
7,credit_card_balance,credit_card_balance.csv,3840312,23,875.69,Convert to Parquet,Main dataset file; will be reused in later not...
8,columns_description,HomeCredit_columns_description.csv,219,5,0.08,Keep CSV,Small helper file; CSV is enough.
9,sample_submission,sample_submission.csv,48744,2,0.74,Keep CSV,Small helper file; CSV is enough.


## 1.3 Data Type and Feature Group Check

In this section, we use the official column description file to understand the feature structure of each table.

The detailed grouping logic is kept inside `group_feature.py` to keep this notebook clean.  
Here, we only generate compact reports showing which tables contain ID columns, financial columns, time columns, categorical columns, payment/balance columns, and other important feature groups.

Expected outputs:

- `feature_group_catalog.csv`
- `feature_group_summary_by_table.csv`
- `feature_group_pivot_by_table.csv`
- `feature_group_table_overview.csv`
- `feature_grouping_validation.csv`
- `feature_group_data_validation.csv`

In [7]:
# Create feature group reports from the official column description file

from group_feature import create_feature_group_reports

feature_reports = create_feature_group_reports(
    description_path=RAW_DATA_DIR / "HomeCredit_columns_description.csv",
    output_dir=OUTPUT_DIR,
    data=data
)

feature_group_overview = feature_reports["table_overview"]
feature_group_pivot = feature_reports["pivot"]
feature_group_validation = feature_reports["grouping_validation"]

feature_group_overview

,source_table,total_columns_in_description,feature_group_count,id_columns,has_target,categorical_columns,numeric_amount_columns,time_offset_columns,binary_flag_columns,count_columns
0,application,122,15,SK_ID_CURR,True,14,10,7,28,2
1,bureau,17,6,"SK_ID_BUREAU, SK_ID_CURR",False,3,3,4,0,1
2,bureau_balance,3,3,SK_ID_BUREAU,False,1,0,1,0,0
3,credit_card_balance,23,9,"SK_ID_CURR, SK_ID_PREV",False,1,13,1,0,1
4,installments_payments,8,4,"SK_ID_CURR, SK_ID_PREV",False,0,2,2,0,2
5,pos_cash_balance,8,5,"SK_ID_CURR, SK_ID_PREV",False,1,0,1,0,2
6,previous_application,38,9,"SK_ID_CURR, SK_ID_PREV",False,15,5,7,4,1


In [8]:
# View file-wise feature group count

feature_group_pivot

feature_group,source_table,applicant_profile_category,binary_flag,building_housing_numeric,bureau_balance_status,bureau_credit_amount,bureau_credit_category,bureau_debt_overdue,bureau_prolong_count,bureau_time_days,...,previous_application_flag,previous_application_rate,previous_application_time_days,region_location,seller_area,social_circle,target,time_days_age,time_hour,time_months
0,application,10.0,2.0,43.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,9.0,0.0,4.0,1.0,6.0,1.0,0.0
1,bureau,0.0,0.0,0.0,0.0,3.0,3.0,4.0,1.0,4.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,bureau_balance,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,credit_card_balance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,installments_payments,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
5,pos_cash_balance,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
6,previous_application,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,4.0,3.0,6.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [9]:
# Check grouping quality

feature_group_validation

,check_name,value,status
0,total_columns_in_description,219,info
1,columns_with_missing_group,0,pass
2,columns_assigned_to_other,0,pass
3,unique_source_tables,7,info
4,unique_feature_groups,40,info


## 1.4 ID and Key Information Check

In this section, we create a compact table-wise ID information report.

The goal is not to make automatic decisions in code.  
We only collect simple facts: which ID columns exist, how many missing values are present in each ID column, and how many duplicated ID values exist.

After viewing this information, we will manually interpret whether the table structure is safe for later aggregation and integration.

Expected output: `id_key_info_report.csv`

In [10]:
# Compact ID information report

id_columns = ["SK_ID_CURR", "SK_ID_PREV", "SK_ID_BUREAU"]

id_info_records = []

for table_name, df in data.items():
    if df is None:
        continue
    
    record = {
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "id_columns_present": ", ".join([col for col in id_columns if col in df.columns])
    }
    
    for id_col in id_columns:
        if id_col in df.columns:
            record[f"missing_{id_col}"] = df[id_col].isna().sum()
            record[f"duplicate_{id_col}"] = df.duplicated(subset=[id_col]).sum()
        else:
            record[f"missing_{id_col}"] = "None"
            record[f"duplicate_{id_col}"] = "None"
    
    id_info_records.append(record)

id_key_info_report = pd.DataFrame(id_info_records)

id_key_info_path = OUTPUT_DIR / "id_key_info_report.csv"
id_key_info_report.to_csv(id_key_info_path, index=False)

id_key_info_report

,table_name,rows,columns,id_columns_present,missing_SK_ID_CURR,duplicate_SK_ID_CURR,missing_SK_ID_PREV,duplicate_SK_ID_PREV,missing_SK_ID_BUREAU,duplicate_SK_ID_BUREAU
0,application_train,307511,122,SK_ID_CURR,0,0,None,None,None,None
1,application_test,48744,121,SK_ID_CURR,0,0,None,None,None,None
2,bureau,1716428,17,"SK_ID_CURR, SK_ID_BUREAU",0,1410617,None,None,0,0
3,bureau_balance,27299925,3,SK_ID_BUREAU,None,None,None,None,0,26482530
4,previous_application,1670214,37,"SK_ID_CURR, SK_ID_PREV",0,1331357,0,0,None,None
5,pos_cash_balance,10001358,8,"SK_ID_CURR, SK_ID_PREV",0,9664106,0,9065033,None,None
6,installments_payments,13605401,8,"SK_ID_CURR, SK_ID_PREV",0,13265814,0,12607649,None,None
7,credit_card_balance,3840312,23,"SK_ID_CURR, SK_ID_PREV",0,3736754,0,3736005,None,None
8,columns_description,219,5,,None,None,None,None,None,None
9,sample_submission,48744,2,SK_ID_CURR,0,0,None,None,None,None


In [11]:
# Check whether train and test applicant IDs overlap

train_ids = set(data["application_train"]["SK_ID_CURR"])
test_ids = set(data["application_test"]["SK_ID_CURR"])

train_test_overlap = len(train_ids & test_ids)

print("Train/Test SK_ID_CURR overlap:", train_test_overlap)

Train/Test SK_ID_CURR overlap: 0


### 1.4 Decision: ID and Key Structure

| Area                             | Finding                                                                                         | Decision                                                                                    |
| -------------------------------- | ----------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------- |
| Application tables               | `application_train` and `application_test` both have unique `SK_ID_CURR` with 0 missing values. | Safe to use as the main applicant-level base tables.                                        |
| Bureau table                     | `SK_ID_BUREAU` is unique, while `SK_ID_CURR` is repeated.                                       | Repeated `SK_ID_CURR` is expected; aggregate bureau records before merging.                 |
| Bureau balance                   | Only `SK_ID_BUREAU` is available and repeated many times.                                       | This is monthly bureau history; aggregate by `SK_ID_BUREAU`, then connect through `bureau`. |
| Previous application             | `SK_ID_PREV` is unique, while `SK_ID_CURR` is repeated.                                         | Safe one-to-many history; aggregate to applicant level before merging.                      |
| POS / installments / credit card | `SK_ID_CURR` and `SK_ID_PREV` have 0 missing values, but both are repeated.                     | Repetition is expected because these are monthly/payment history tables.                    |
| Helper files                     | Column description and sample submission are not modeling history tables.                       | Use only for metadata and final submission format.                                          |

**Final decision:** The ID structure is safe for data engineering. No raw historical table should be directly merged with application data. All one-to-many tables must be aggregated first so the final dataset remains one row per `SK_ID_CURR`.


## 1.5 Basic Anomaly Information Check

In this section, we only collect basic anomaly-related information from the raw tables.

We do not clean, remove, or modify any value here.  
The goal is to identify obvious unusual values such as placeholder day values, unknown category labels, negative amount/count values, and zero financial values.

The interpretation and cleaning decisions will be written separately after reviewing the output.

Expected outputs:

- `anomaly_info_report.csv`
- `anomaly_summary_by_table.csv`

In [12]:
# Basic anomaly information report

anomaly_records = []

for table_name, df in data.items():
    if df is None:
        continue
    
    # day placeholder values
    day_cols = [col for col in df.columns if col.startswith("DAYS_")]
    for col in day_cols:
        count = (df[col] == 365243).sum()
        if count > 0:
            anomaly_records.append({
                "table_name": table_name,
                "column_name": col,
                "anomaly_type": "placeholder_365243",
                "count": count,
                "percent": round(count / len(df) * 100, 4)
            })
    
    # unknown categorical labels
    cat_cols = df.select_dtypes(include="object").columns
    for col in cat_cols:
        values = df[col].astype(str).str.upper()
        count = values.isin(["XNA", "UNKNOWN", "NAN"]).sum()
        if count > 0:
            anomaly_records.append({
                "table_name": table_name,
                "column_name": col,
                "anomaly_type": "unknown_category_label",
                "count": count,
                "percent": round(count / len(df) * 100, 4)
            })
    
    # negative amount values
    amount_cols = [col for col in df.columns if col.startswith("AMT_")]
    for col in amount_cols:
        count = (df[col] < 0).sum()
        if count > 0:
            anomaly_records.append({
                "table_name": table_name,
                "column_name": col,
                "anomaly_type": "negative_amount_value",
                "count": count,
                "percent": round(count / len(df) * 100, 4)
            })
    
    # zero amount values
    for col in amount_cols:
        count = (df[col] == 0).sum()
        if count > 0:
            anomaly_records.append({
                "table_name": table_name,
                "column_name": col,
                "anomaly_type": "zero_amount_value",
                "count": count,
                "percent": round(count / len(df) * 100, 4)
            })
    
    # negative count values
    count_cols = [col for col in df.columns if col.startswith(("CNT_", "NUM_"))]
    for col in count_cols:
        count = (df[col] < 0).sum()
        if count > 0:
            anomaly_records.append({
                "table_name": table_name,
                "column_name": col,
                "anomaly_type": "negative_count_value",
                "count": count,
                "percent": round(count / len(df) * 100, 4)
            })

anomaly_info_report = pd.DataFrame(anomaly_records)

anomaly_summary_by_table = (
    anomaly_info_report
    .groupby(["table_name", "anomaly_type"], as_index=False)["count"]
    .sum()
)

anomaly_info_report.to_csv(OUTPUT_DIR / "anomaly_info_report.csv", index=False)
anomaly_summary_by_table.to_csv(OUTPUT_DIR / "anomaly_summary_by_table.csv", index=False)

anomaly_info_report

,table_name,column_name,anomaly_type,count,percent
0,application_train,DAYS_EMPLOYED,placeholder_365243,55374,18.0072
1,application_train,CODE_GENDER,unknown_category_label,4,0.0013
2,application_train,NAME_TYPE_SUITE,unknown_category_label,1292,0.4201
3,application_train,NAME_FAMILY_STATUS,unknown_category_label,2,0.0007
4,application_train,OCCUPATION_TYPE,unknown_category_label,96391,31.3455
...,...,...,...,...,...
78,credit_card_balance,AMT_PAYMENT_TOTAL_CURRENT,zero_amount_value,2172223,56.5637
79,credit_card_balance,AMT_RECEIVABLE_PRINCIPAL,zero_amount_value,2296167,59.7912
80,credit_card_balance,AMT_RECIVABLE,zero_amount_value,2113816,55.0428
81,credit_card_balance,AMT_TOTAL_RECEIVABLE,zero_amount_value,2113643,55.0383


In [13]:
anomaly_summary_by_table

,table_name,anomaly_type,count
0,application_test,placeholder_365243,9274
1,application_test,unknown_category_label,128308
2,application_test,zero_amount_value,204549
3,application_train,placeholder_365243,55374
4,application_train,unknown_category_label,819751
5,application_train,zero_amount_value,1295776
6,bureau,negative_amount_value,8769
7,bureau,zero_amount_value,4572993
8,columns_description,unknown_category_label,133
9,credit_card_balance,negative_amount_value,223445


### 1.5 Decision: Basic Anomaly Information

| Area                    | Observation                                                                                                     | Decision                                                                                 |
| ----------------------- | --------------------------------------------------------------------------------------------------------------- | ---------------------------------------------------------------------------------------- |
| Day placeholder         | `365243` appears in `DAYS_EMPLOYED` and several previous-application day columns.                               | Treat as a special placeholder, not a real day value.                                    |
| Unknown category labels | `XNA`, `Unknown`, and similar labels appear mainly in application and previous-application categorical columns. | Keep for now; handle later during categorical cleaning/encoding.                         |
| Zero amount values      | Many `AMT_` columns contain zero values, especially in bureau and credit-card balance tables.                   | Do not remove directly; zero can be meaningful for inactive/no-balance/no-payment cases. |
| Negative amount values  | Some negative values appear in bureau, previous application, and credit-card balance amount columns.            | Review during feature cleaning before final modeling.                                    |
| Metadata file           | `columns_description` shows unknown labels, but it is not a modeling table.                                     | Ignore for anomaly cleaning.                                                             |

**Final decision:** No values will be changed in this step. The anomaly report will only guide later cleaning and feature engineering, especially for `365243`, unknown categorical labels, zero financial values, and negative amount values.


## 1.6 Train/Test Consistency Check

In this section, we compare `application_train` and `application_test`.

The goal is to collect simple structural information: column consistency, dtype consistency, missing-value percentage differences, and categorical level differences between train and test.

No cleaning, dropping, encoding, or modeling decision is made in this step.
The output will guide later preprocessing so that train and test datasets remain aligned.


In [14]:
# Train/test column and dtype information

train_df = data["application_train"]
test_df = data["application_test"]

train_features = set(train_df.columns) - {"TARGET"}
test_features = set(test_df.columns)
common_features = sorted(train_features & test_features)

train_only_features = sorted(train_features - test_features)
test_only_features = sorted(test_features - train_features)

train_test_column_info = pd.DataFrame([
    {
        "train_columns_total": train_df.shape[1],
        "test_columns_total": test_df.shape[1],
        "train_features_without_TARGET": len(train_features),
        "test_features": len(test_features),
        "common_features": len(common_features),
        "train_only_features_count": len(train_only_features),
        "test_only_features_count": len(test_only_features),
        "train_only_features": ", ".join(train_only_features) if train_only_features else "None",
        "test_only_features": ", ".join(test_only_features) if test_only_features else "None",
        "TARGET_in_train": "TARGET" in train_df.columns,
        "TARGET_in_test": "TARGET" in test_df.columns,
    }
])

dtype_info = pd.DataFrame({
    "column_name": common_features,
    "train_dtype": [str(train_df[col].dtype) for col in common_features],
    "test_dtype": [str(test_df[col].dtype) for col in common_features],
})

dtype_mismatch_info = dtype_info[
    dtype_info["train_dtype"] != dtype_info["test_dtype"]
].reset_index(drop=True)

train_test_column_info.to_csv(OUTPUT_DIR / "train_test_column_info.csv", index=False)
dtype_mismatch_info.to_csv(OUTPUT_DIR / "train_test_dtype_mismatch_info.csv", index=False)

train_test_column_info

,train_columns_total,test_columns_total,train_features_without_TARGET,test_features,common_features,train_only_features_count,test_only_features_count,train_only_features,test_only_features,TARGET_in_train,TARGET_in_test
0,122,121,121,121,121,0,0,None,None,True,False


In [15]:
# Train/test missing percentage difference

missing_diff_info = pd.DataFrame({
    "column_name": common_features,
    "train_missing_pct": [round(train_df[col].isna().mean() * 100, 4) for col in common_features],
    "test_missing_pct": [round(test_df[col].isna().mean() * 100, 4) for col in common_features],
})

missing_diff_info["absolute_difference_pct"] = (
    missing_diff_info["train_missing_pct"] - missing_diff_info["test_missing_pct"]
).abs()

missing_diff_info = missing_diff_info.sort_values(
    "absolute_difference_pct", ascending=False
).reset_index(drop=True)

missing_diff_info.to_csv(
    OUTPUT_DIR / "train_test_missing_difference_info.csv",
    index=False
)

missing_diff_info.head(20)

,column_name,train_missing_pct,test_missing_pct,absolute_difference_pct
0,EXT_SOURCE_1,56.3811,42.1221,14.2590
1,EXT_SOURCE_3,19.8253,17.7827,2.0426
2,ENTRANCES_AVG,50.3488,48.3731,1.9757
3,ENTRANCES_MEDI,50.3488,48.3731,1.9757
4,ENTRANCES_MODE,50.3488,48.3731,1.9757
5,FLOORSMAX_MODE,49.7608,47.8438,1.9170
6,FLOORSMAX_MEDI,49.7608,47.8438,1.9170
7,FLOORSMAX_AVG,49.7608,47.8438,1.9170
8,YEARS_BEGINEXPLUATATION_MODE,48.7810,46.8899,1.8911
9,YEARS_BEGINEXPLUATATION_AVG,48.7810,46.8899,1.8911


In [16]:
# Train/test categorical level information

categorical_common_cols = [
    col for col in common_features
    if train_df[col].dtype == "object" and test_df[col].dtype == "object"
]

category_records = []

for col in categorical_common_cols:
    train_levels = set(train_df[col].dropna().astype(str).unique())
    test_levels = set(test_df[col].dropna().astype(str).unique())
    
    train_only_levels = sorted(train_levels - test_levels)
    test_only_levels = sorted(test_levels - train_levels)
    
    category_records.append({
        "column_name": col,
        "train_unique_categories": len(train_levels),
        "test_unique_categories": len(test_levels),
        "train_only_category_count": len(train_only_levels),
        "test_only_category_count": len(test_only_levels),
        "train_only_category_examples": ", ".join(train_only_levels[:10]) if train_only_levels else "None",
        "test_only_category_examples": ", ".join(test_only_levels[:10]) if test_only_levels else "None",
    })

train_test_category_info = pd.DataFrame(category_records)

train_test_category_info.to_csv(
    OUTPUT_DIR / "train_test_category_level_info.csv",
    index=False
)

train_test_category_info

,column_name,train_unique_categories,test_unique_categories,train_only_category_count,test_only_category_count,train_only_category_examples,test_only_category_examples
0,CODE_GENDER,3,2,1,0,XNA,None
1,EMERGENCYSTATE_MODE,2,2,0,0,None,None
2,FLAG_OWN_CAR,2,2,0,0,None,None
3,FLAG_OWN_REALTY,2,2,0,0,None,None
4,FONDKAPREMONT_MODE,4,4,0,0,None,None
5,HOUSETYPE_MODE,3,3,0,0,None,None
6,NAME_CONTRACT_TYPE,2,2,0,0,None,None
7,NAME_EDUCATION_TYPE,5,5,0,0,None,None
8,NAME_FAMILY_STATUS,6,5,1,0,Unknown,None
9,NAME_HOUSING_TYPE,6,6,0,0,None,None


### 1.6 Decision: Train/Test Consistency

| Area                          | Observation                                                                                                 | Decision                                                                            |
| ----------------------------- | ----------------------------------------------------------------------------------------------------------- | ----------------------------------------------------------------------------------- |
| Column structure              | `application_train` has one extra column: `TARGET`. After removing `TARGET`, train and test features align. | Use `application_train` as training base and `application_test` as prediction base. |
| Target column                 | `TARGET` is available only in train and absent in test.                                                     | This is expected for competition-style supervised learning.                         |
| Common features               | Train and test share the same applicant-level feature structure after excluding `TARGET`.                   | Future preprocessing should be applied using the same common feature list.          |
| Data types                    | No major structural dtype issue was found from the dtype comparison report.                                 | Keep dtype conversion decisions for the preprocessing stage.                        |
| Missing percentage difference | Some columns may have different missing rates between train and test.                                       | Do not drop features here; review missingness during EDA and imputation planning.   |
| Categorical levels            | Some categorical columns may contain train-only or test-only levels.                                        | Use encoding methods that can safely handle unseen categories.                      |

**Final decision:** Train and test application files are structurally aligned for data engineering. No feature will be removed in this step. Later preprocessing must keep train/test columns consistent and handle missing values and categorical levels carefully.


## Step 2: Applicant-Level Integration

In this step, we convert raw historical tables into applicant-level feature blocks.

The final dataset must keep one fixed grain:

| Final grain                          | Meaning                                                                 |
| ------------------------------------ | ----------------------------------------------------------------------- |
| `1 SK_ID_CURR = 1 applicant = 1 row` | Each applicant should appear only once in the final train/test dataset. |

Raw historical tables will not be merged directly with the application base.
Each table will first be summarized into an `SK_ID_CURR`-level feature block, then merged with `application_train` and `application_test`.


## 2.1 Bureau and Bureau Balance Integration

This section creates the external credit history feature block from `bureau.csv` and `bureau_balance.csv`.

The relationship path is:

```text
bureau_balance → aggregate by SK_ID_BUREAU → merge with bureau → aggregate by SK_ID_CURR → bureau_features
```

`bureau` contains external credit records for applicants.
`bureau_balance` contains monthly status history for those external bureau credits.

The important point is that `bureau_balance` does not contain `SK_ID_CURR`.
So it cannot be connected directly with the application base. It must first be summarized by `SK_ID_BUREAU`, then merged with `bureau`, and finally summarized by `SK_ID_CURR`.

| Step  | Work                                                                   |
| ----- | ---------------------------------------------------------------------- |
| 2.1.1 | Collect relationship information between `bureau` and `bureau_balance` |
| 2.1.2 | Inspect `bureau_balance` status and month information                  |
| 2.1.3 | Create `bureau_balance` summary at `SK_ID_BUREAU` level                |
| 2.1.4 | Merge bureau balance summary into `bureau`                             |
| 2.1.5 | Create final `bureau_features` at `SK_ID_CURR` level                   |
| 2.1.6 | Check final bureau feature block shape and key uniqueness              |
| 2.1.7 | Save the bureau feature block                                          |

No raw bureau or bureau balance rows will be directly merged with the application base.


### 2.1.1 Bureau Relationship Information

In this section, we collect basic relationship information from `bureau` and `bureau_balance`.

We only check simple facts such as row count, available ID columns, missing key values, unique ID counts, and how many `SK_ID_BUREAU` values are shared between the two tables.

No aggregation or cleaning is done in this step.


In [17]:
# Bureau and bureau_balance relationship information

bureau = data["bureau"]
bureau_balance = data["bureau_balance"]

def id_info(df, table_name):
    id_cols = ["SK_ID_CURR", "SK_ID_BUREAU"]
    
    record = {
        "table_name": table_name,
        "rows": df.shape[0],
        "columns": df.shape[1],
    }
    
    for col in id_cols:
        if col in df.columns:
            record[f"{col}_missing"] = df[col].isna().sum()
            record[f"{col}_unique"] = df[col].nunique()
            record[f"{col}_duplicate_values"] = df.duplicated(subset=[col]).sum()
        else:
            record[f"{col}_missing"] = "None"
            record[f"{col}_unique"] = "None"
            record[f"{col}_duplicate_values"] = "None"
    
    return record

bureau_relation_info = pd.DataFrame([
    id_info(bureau, "bureau"),
    id_info(bureau_balance, "bureau_balance")
])

bureau_ids = set(bureau["SK_ID_BUREAU"].dropna().unique())
bureau_balance_ids = set(bureau_balance["SK_ID_BUREAU"].dropna().unique())

bureau_connection_info = pd.DataFrame([
    {
        "bureau_unique_SK_ID_BUREAU": len(bureau_ids),
        "bureau_balance_unique_SK_ID_BUREAU": len(bureau_balance_ids),
        "common_SK_ID_BUREAU": len(bureau_ids & bureau_balance_ids),
        "bureau_only_SK_ID_BUREAU": len(bureau_ids - bureau_balance_ids),
        "bureau_balance_only_SK_ID_BUREAU": len(bureau_balance_ids - bureau_ids),
    }
])

bureau_relation_info.to_csv(OUTPUT_DIR / "bureau_relation_info.csv", index=False)
bureau_connection_info.to_csv(OUTPUT_DIR / "bureau_connection_info.csv", index=False)

bureau_relation_info

,table_name,rows,columns,SK_ID_CURR_missing,SK_ID_CURR_unique,SK_ID_CURR_duplicate_values,SK_ID_BUREAU_missing,SK_ID_BUREAU_unique,SK_ID_BUREAU_duplicate_values
0,bureau,1716428,17,0,305811,1410617,0,1716428,0
1,bureau_balance,27299925,3,None,None,None,0,817395,26482530


### 2.1.2 Bureau Balance Status and Month Information

In this section, we inspect the monthly bureau balance table before aggregation.

We collect simple information about `STATUS`, `MONTHS_BALANCE`, and the number of monthly records available for each `SK_ID_BUREAU`.

This helps us understand what type of summary can be created later at `SK_ID_BUREAU` level.


In [18]:
# Bureau balance status and month information

bureau_balance = data["bureau_balance"]

bureau_balance_status_info = (
    bureau_balance["STATUS"]
    .value_counts(dropna=False)
    .reset_index()
)

bureau_balance_status_info.columns = ["STATUS", "count"]
bureau_balance_status_info["percent"] = round(
    bureau_balance_status_info["count"] / len(bureau_balance) * 100, 4
)

bureau_balance_month_info = pd.DataFrame([
    {
        "rows": bureau_balance.shape[0],
        "unique_SK_ID_BUREAU": bureau_balance["SK_ID_BUREAU"].nunique(),
        "min_MONTHS_BALANCE": bureau_balance["MONTHS_BALANCE"].min(),
        "max_MONTHS_BALANCE": bureau_balance["MONTHS_BALANCE"].max(),
        "unique_MONTHS_BALANCE": bureau_balance["MONTHS_BALANCE"].nunique(),
        "missing_STATUS": bureau_balance["STATUS"].isna().sum(),
        "missing_MONTHS_BALANCE": bureau_balance["MONTHS_BALANCE"].isna().sum(),
    }
])

bureau_balance_history_length_info = (
    bureau_balance
    .groupby("SK_ID_BUREAU")
    .size()
    .describe()
    .reset_index()
)

bureau_balance_history_length_info.columns = ["metric", "monthly_record_count"]

bureau_balance_status_info.to_csv(
    OUTPUT_DIR / "bureau_balance_status_info.csv",
    index=False
)

bureau_balance_month_info.to_csv(
    OUTPUT_DIR / "bureau_balance_month_info.csv",
    index=False
)

bureau_balance_history_length_info.to_csv(
    OUTPUT_DIR / "bureau_balance_history_length_info.csv",
    index=False
)

bureau_balance_status_info

,STATUS,count,percent
0,C,13646993,49.9891
1,0,7499507,27.4708
2,X,5810482,21.2839
3,1,242347,0.8877
4,5,62406,0.2286
5,2,23419,0.0858
6,3,8924,0.0327
7,4,5847,0.0214


In [19]:
bureau_balance_month_info

,rows,unique_SK_ID_BUREAU,min_MONTHS_BALANCE,max_MONTHS_BALANCE,unique_MONTHS_BALANCE,missing_STATUS,missing_MONTHS_BALANCE
0,27299925,817395,-96,0,97,0,0


In [20]:
bureau_balance_history_length_info

,metric,monthly_record_count
0,count,817395.000000
1,mean,33.398693
2,std,25.794666
3,min,1.000000
4,25%,13.000000
5,50%,26.000000
6,75%,48.000000
7,max,97.000000


### 2.1.3 Bureau Balance Aggregation by SK_ID_BUREAU

In this step, we summarize `bureau_balance` from monthly status-level data to bureau-credit-level data.

The output table will have one row per `SK_ID_BUREAU`.

This table is not applicant-level yet.
It will be merged into `bureau` first, and then the enriched bureau table will be summarized by `SK_ID_CURR`.


In [21]:
# Aggregate bureau_balance to one row per SK_ID_BUREAU

import numpy as np

bb = data["bureau_balance"][["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"]].copy()

bb["STATUS"] = bb["STATUS"].astype(str)

status_order = ["0", "1", "2", "3", "4", "5", "C", "X"]

bb["late_level"] = bb["STATUS"].map({
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "C": 0,
    "X": 0,
}).fillna(0)

bb["is_late"] = bb["late_level"].between(1, 5).astype(int)
bb["is_severe_late"] = bb["late_level"].between(3, 5).astype(int)

bb["is_recent_6m"] = (bb["MONTHS_BALANCE"] >= -6).astype(int)
bb["is_recent_12m"] = (bb["MONTHS_BALANCE"] >= -12).astype(int)
bb["is_old"] = (bb["MONTHS_BALANCE"] < -12).astype(int)

In [22]:
# Basic month and risk summary features

month_features = (
    bb.groupby("SK_ID_BUREAU")
    .agg(
        bureau_bal_month_count=("MONTHS_BALANCE", "count"),
        bureau_bal_month_min=("MONTHS_BALANCE", "min"),
        bureau_bal_month_max=("MONTHS_BALANCE", "max"),
        bureau_bal_month_mean=("MONTHS_BALANCE", "mean"),
        bureau_bal_unique_status_count=("STATUS", "nunique"),
        bureau_bal_late_count=("is_late", "sum"),
        bureau_bal_severe_late_count=("is_severe_late", "sum"),
        bureau_bal_max_late_status=("late_level", "max"),
    )
    .reset_index()
)

month_features["bureau_bal_month_span"] = (
    month_features["bureau_bal_month_max"] - month_features["bureau_bal_month_min"]
)

month_features["bureau_bal_late_ratio"] = (
    month_features["bureau_bal_late_count"] / month_features["bureau_bal_month_count"]
)

month_features["bureau_bal_severe_late_ratio"] = (
    month_features["bureau_bal_severe_late_count"] / month_features["bureau_bal_month_count"]
)

month_features["bureau_bal_has_late"] = (month_features["bureau_bal_late_count"] > 0).astype(int)
month_features["bureau_bal_has_severe_late"] = (month_features["bureau_bal_severe_late_count"] > 0).astype(int)

In [23]:
# Status count and ratio features

status_counts = pd.crosstab(bb["SK_ID_BUREAU"], bb["STATUS"])

for status in status_order:
    if status not in status_counts.columns:
        status_counts[status] = 0

status_counts = status_counts[status_order]
status_counts.columns = [f"bureau_bal_status_{status}_count" for status in status_counts.columns]
status_counts = status_counts.reset_index()

bureau_balance_by_bureau = month_features.merge(
    status_counts,
    on="SK_ID_BUREAU",
    how="left"
)

for status in status_order:
    count_col = f"bureau_bal_status_{status}_count"
    ratio_col = f"bureau_bal_status_{status}_ratio"
    
    bureau_balance_by_bureau[ratio_col] = (
        bureau_balance_by_bureau[count_col] / bureau_balance_by_bureau["bureau_bal_month_count"]
    )

In [24]:
# Recent, old, and trend-based risk features

recent_6m = (
    bb[bb["is_recent_6m"] == 1]
    .groupby("SK_ID_BUREAU")
    .agg(
        bureau_bal_recent_6m_count=("MONTHS_BALANCE", "count"),
        bureau_bal_recent_6m_late_count=("is_late", "sum"),
        bureau_bal_recent_6m_severe_late_count=("is_severe_late", "sum"),
    )
    .reset_index()
)

recent_12m = (
    bb[bb["is_recent_12m"] == 1]
    .groupby("SK_ID_BUREAU")
    .agg(
        bureau_bal_recent_12m_count=("MONTHS_BALANCE", "count"),
        bureau_bal_recent_12m_late_count=("is_late", "sum"),
        bureau_bal_recent_12m_severe_late_count=("is_severe_late", "sum"),
    )
    .reset_index()
)

old_history = (
    bb[bb["is_old"] == 1]
    .groupby("SK_ID_BUREAU")
    .agg(
        bureau_bal_old_count=("MONTHS_BALANCE", "count"),
        bureau_bal_old_late_count=("is_late", "sum"),
    )
    .reset_index()
)

bureau_balance_by_bureau = (
    bureau_balance_by_bureau
    .merge(recent_6m, on="SK_ID_BUREAU", how="left")
    .merge(recent_12m, on="SK_ID_BUREAU", how="left")
    .merge(old_history, on="SK_ID_BUREAU", how="left")
)

new_count_cols = [
    col for col in bureau_balance_by_bureau.columns
    if col.endswith("_count")
]

bureau_balance_by_bureau[new_count_cols] = bureau_balance_by_bureau[new_count_cols].fillna(0)

bureau_balance_by_bureau["bureau_bal_recent_6m_late_ratio"] = (
    bureau_balance_by_bureau["bureau_bal_recent_6m_late_count"]
    / bureau_balance_by_bureau["bureau_bal_recent_6m_count"].replace(0, np.nan)
)

bureau_balance_by_bureau["bureau_bal_recent_12m_late_ratio"] = (
    bureau_balance_by_bureau["bureau_bal_recent_12m_late_count"]
    / bureau_balance_by_bureau["bureau_bal_recent_12m_count"].replace(0, np.nan)
)

bureau_balance_by_bureau["bureau_bal_recent_12m_severe_late_ratio"] = (
    bureau_balance_by_bureau["bureau_bal_recent_12m_severe_late_count"]
    / bureau_balance_by_bureau["bureau_bal_recent_12m_count"].replace(0, np.nan)
)

bureau_balance_by_bureau["bureau_bal_old_late_ratio"] = (
    bureau_balance_by_bureau["bureau_bal_old_late_count"]
    / bureau_balance_by_bureau["bureau_bal_old_count"].replace(0, np.nan)
)

bureau_balance_by_bureau["bureau_bal_late_trend_12m_vs_old"] = (
    bureau_balance_by_bureau["bureau_bal_recent_12m_late_ratio"]
    - bureau_balance_by_bureau["bureau_bal_old_late_ratio"]
)

ratio_cols = [
    col for col in bureau_balance_by_bureau.columns
    if col.endswith("_ratio") or "trend" in col
]

bureau_balance_by_bureau[ratio_cols] = bureau_balance_by_bureau[ratio_cols].fillna(0)

In [25]:
# Bureau balance aggregation info

bureau_balance_by_bureau_info = pd.DataFrame([
    {
        "rows": bureau_balance_by_bureau.shape[0],
        "columns": bureau_balance_by_bureau.shape[1],
        "unique_SK_ID_BUREAU": bureau_balance_by_bureau["SK_ID_BUREAU"].nunique(),
        "duplicate_SK_ID_BUREAU": bureau_balance_by_bureau.duplicated("SK_ID_BUREAU").sum(),
    }
])

bureau_balance_by_bureau.to_csv(
    OUTPUT_DIR / "bureau_balance_by_bureau.csv",
    index=False
)

bureau_balance_by_bureau_info.to_csv(
    OUTPUT_DIR / "bureau_balance_by_bureau_info.csv",
    index=False
)

bureau_balance_by_bureau.head()

,SK_ID_BUREAU,bureau_bal_month_count,bureau_bal_month_min,bureau_bal_month_max,bureau_bal_month_mean,bureau_bal_unique_status_count,bureau_bal_late_count,bureau_bal_severe_late_count,bureau_bal_max_late_status,bureau_bal_month_span,...,bureau_bal_recent_12m_count,bureau_bal_recent_12m_late_count,bureau_bal_recent_12m_severe_late_count,bureau_bal_old_count,bureau_bal_old_late_count,bureau_bal_recent_6m_late_ratio,bureau_bal_recent_12m_late_ratio,bureau_bal_recent_12m_severe_late_ratio,bureau_bal_old_late_ratio,bureau_bal_late_trend_12m_vs_old
0,5001709,97,-96,0,-48.0,2,0,0,0,96,...,13.0,0.0,0.0,84.0,0.0,0.0,0.0,0.0,0.0,0.0
1,5001710,83,-82,0,-41.0,3,0,0,0,82,...,13.0,0.0,0.0,70.0,0.0,0.0,0.0,0.0,0.0,0.0
2,5001711,4,-3,0,-1.5,2,0,0,0,3,...,4.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,5001712,19,-18,0,-9.0,2,0,0,0,18,...,13.0,0.0,0.0,6.0,0.0,0.0,0.0,0.0,0.0,0.0
4,5001713,22,-21,0,-10.5,1,0,0,0,21,...,13.0,0.0,0.0,9.0,0.0,0.0,0.0,0.0,0.0,0.0


In [26]:
bureau_balance_by_bureau_info

,rows,columns,unique_SK_ID_BUREAU,duplicate_SK_ID_BUREAU
0,817395,43,817395,0


In [27]:
# Merge bureau_balance summary with bureau

bureau = data["bureau"].copy()

bureau_enriched = bureau.merge(
    bureau_balance_by_bureau,
    on="SK_ID_BUREAU",
    how="left"
)

bureau_enriched_info = pd.DataFrame([
    {
        "bureau_rows_before_merge": bureau.shape[0],
        "bureau_columns_before_merge": bureau.shape[1],
        "bureau_enriched_rows": bureau_enriched.shape[0],
        "bureau_enriched_columns": bureau_enriched.shape[1],
        "unique_SK_ID_CURR": bureau_enriched["SK_ID_CURR"].nunique(),
        "unique_SK_ID_BUREAU": bureau_enriched["SK_ID_BUREAU"].nunique(),
        "duplicate_SK_ID_BUREAU": bureau_enriched.duplicated("SK_ID_BUREAU").sum(),
        "matched_bureau_balance_rows": bureau_enriched["bureau_bal_month_count"].notna().sum(),
        "unmatched_bureau_balance_rows": bureau_enriched["bureau_bal_month_count"].isna().sum(),
    }
])

bureau_enriched.to_csv(
    OUTPUT_DIR / "bureau_enriched.csv",
    index=False
)

bureau_enriched_info.to_csv(
    OUTPUT_DIR / "bureau_enriched_info.csv",
    index=False
)

bureau_enriched_info

,bureau_rows_before_merge,bureau_columns_before_merge,bureau_enriched_rows,bureau_enriched_columns,unique_SK_ID_CURR,unique_SK_ID_BUREAU,duplicate_SK_ID_BUREAU,matched_bureau_balance_rows,unmatched_bureau_balance_rows
0,1716428,17,1716428,59,305811,1716428,0,774354,942074


In [28]:
bureau_enriched.head()

,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,...,bureau_bal_recent_12m_count,bureau_bal_recent_12m_late_count,bureau_bal_recent_12m_severe_late_count,bureau_bal_old_count,bureau_bal_old_late_count,bureau_bal_recent_6m_late_ratio,bureau_bal_recent_12m_late_ratio,bureau_bal_recent_12m_severe_late_ratio,bureau_bal_old_late_ratio,bureau_bal_late_trend_12m_vs_old
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,215354,5714464,Active,currency 1,-203,0,528.0,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,215354,5714465,Active,currency 1,-203,0,NaN,NaN,NaN,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,215354,5714466,Active,currency 1,-629,0,1197.0,NaN,77674.5,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 2.1.5 Create Applicant-Level Bureau Feature Block

In this step, we convert `bureau_enriched` into an applicant-level feature block.

Current structure:

```text
bureau_enriched
many rows per SK_ID_CURR
```

Target structure:

```text
bureau_features
1 row per SK_ID_CURR
```

This feature block will summarize external credit history, current debt pressure, overdue risk, credit recency, active/closed credit behavior, selected credit-type patterns, bureau-balance monthly risk, and latest bureau-credit information.

No target variable is used in this step.


In [29]:
# Prepare bureau_enriched for applicant-level aggregation

import numpy as np

be = bureau_enriched.copy()

be["CREDIT_ACTIVE"] = be["CREDIT_ACTIVE"].astype(str)
be["CREDIT_TYPE"] = be["CREDIT_TYPE"].astype(str)
be["CREDIT_CURRENCY"] = be["CREDIT_CURRENCY"].astype(str)

be["bureau_is_active"] = be["CREDIT_ACTIVE"].str.lower().eq("active").astype(int)
be["bureau_is_closed"] = be["CREDIT_ACTIVE"].str.lower().eq("closed").astype(int)
be["bureau_is_sold"] = be["CREDIT_ACTIVE"].str.lower().eq("sold").astype(int)
be["bureau_is_bad_debt"] = be["CREDIT_ACTIVE"].str.lower().str.contains("bad", na=False).astype(int)

be["bureau_has_overdue_days"] = (be["CREDIT_DAY_OVERDUE"].fillna(0) > 0).astype(int)
be["bureau_has_overdue_amount"] = (be["AMT_CREDIT_SUM_OVERDUE"].fillna(0) > 0).astype(int)
be["bureau_has_debt"] = (be["AMT_CREDIT_SUM_DEBT"].fillna(0) > 0).astype(int)

be["bureau_has_future_enddate"] = (be["DAYS_CREDIT_ENDDATE"].fillna(0) > 0).astype(int)

be["bureau_is_recent_6m"] = (be["DAYS_CREDIT"] >= -180).astype(int)
be["bureau_is_recent_12m"] = (be["DAYS_CREDIT"] >= -365).astype(int)
be["bureau_is_recent_24m"] = (be["DAYS_CREDIT"] >= -730).astype(int)

be["bureau_recent_update_6m"] = (be["DAYS_CREDIT_UPDATE"] >= -180).astype(int)
be["bureau_recent_update_12m"] = (be["DAYS_CREDIT_UPDATE"] >= -365).astype(int)

be["bureau_debt_to_credit"] = (
    be["AMT_CREDIT_SUM_DEBT"] / be["AMT_CREDIT_SUM"].replace(0, np.nan)
)

be["bureau_overdue_to_credit"] = (
    be["AMT_CREDIT_SUM_OVERDUE"] / be["AMT_CREDIT_SUM"].replace(0, np.nan)
)

be["bureau_annuity_to_credit"] = (
    be["AMT_ANNUITY"] / be["AMT_CREDIT_SUM"].replace(0, np.nan)
)

be["bureau_debt_to_limit"] = (
    be["AMT_CREDIT_SUM_DEBT"] / be["AMT_CREDIT_SUM_LIMIT"].replace(0, np.nan)
)

be["bureau_enddate_gap"] = (
    be["DAYS_ENDDATE_FACT"] - be["DAYS_CREDIT_ENDDATE"]
)

if "bureau_bal_month_count" in be.columns:
    be["bureau_has_balance_history"] = be["bureau_bal_month_count"].notna().astype(int)
else:
    be["bureau_has_balance_history"] = 0

In [30]:
# Selected credit type pattern features

selected_credit_types = {
    "consumer_credit": "Consumer credit",
    "credit_card": "Credit card",
    "car_loan": "Car loan",
    "mortgage": "Mortgage",
    "microloan": "Microloan",
    "business_loan": "Loan for business development",
    "working_capital_loan": "Loan for working capital replenishment",
}

selected_type_values = list(selected_credit_types.values())

for feature_name, credit_type_value in selected_credit_types.items():
    be[f"bureau_type_{feature_name}"] = (
        be["CREDIT_TYPE"].eq(credit_type_value).astype(int)
    )

be["bureau_type_other"] = (
    ~be["CREDIT_TYPE"].isin(selected_type_values)
).astype(int)

In [31]:
# Basic applicant-level bureau features

bureau_features = (
    be.groupby("SK_ID_CURR")
    .agg(
        bureau_record_count=("SK_ID_BUREAU", "count"),
        bureau_unique_credit_type_count=("CREDIT_TYPE", "nunique"),
        bureau_unique_currency_count=("CREDIT_CURRENCY", "nunique"),
    )
    .reset_index()
)

flag_name_map = {
    "bureau_is_active": "bureau_active",
    "bureau_is_closed": "bureau_closed",
    "bureau_is_sold": "bureau_sold",
    "bureau_is_bad_debt": "bureau_bad_debt",
    "bureau_has_overdue_days": "bureau_overdue_days_record",
    "bureau_has_overdue_amount": "bureau_overdue_amount_record",
    "bureau_has_debt": "bureau_debt_record",
    "bureau_has_future_enddate": "bureau_future_enddate",
    "bureau_is_recent_6m": "bureau_recent_6m",
    "bureau_is_recent_12m": "bureau_recent_12m",
    "bureau_is_recent_24m": "bureau_recent_24m",
    "bureau_recent_update_6m": "bureau_recent_update_6m",
    "bureau_recent_update_12m": "bureau_recent_update_12m",
    "bureau_has_balance_history": "bureau_with_balance",
}

for col, prefix in flag_name_map.items():
    temp = (
        be.groupby("SK_ID_CURR")[col]
        .agg(["sum", "mean"])
        .reset_index()
    )
    temp.columns = ["SK_ID_CURR", f"{prefix}_count", f"{prefix}_ratio"]
    bureau_features = bureau_features.merge(temp, on="SK_ID_CURR", how="left")

type_flag_cols = [f"bureau_type_{name}" for name in selected_credit_types.keys()]
type_flag_cols.append("bureau_type_other")

for col in type_flag_cols:
    temp = (
        be.groupby("SK_ID_CURR")[col]
        .agg(["sum", "mean"])
        .reset_index()
    )
    temp.columns = ["SK_ID_CURR", f"{col}_count", f"{col}_ratio"]
    bureau_features = bureau_features.merge(temp, on="SK_ID_CURR", how="left")

bureau_features["bureau_record_per_credit_type_avg"] = (
    bureau_features["bureau_record_count"]
    / bureau_features["bureau_unique_credit_type_count"].replace(0, np.nan)
)

bureau_features.head()

,SK_ID_CURR,bureau_record_count,bureau_unique_credit_type_count,bureau_unique_currency_count,bureau_active_count,bureau_active_ratio,bureau_closed_count,bureau_closed_ratio,bureau_sold_count,bureau_sold_ratio,...,bureau_type_mortgage_ratio,bureau_type_microloan_count,bureau_type_microloan_ratio,bureau_type_business_loan_count,bureau_type_business_loan_ratio,bureau_type_working_capital_loan_count,bureau_type_working_capital_loan_ratio,bureau_type_other_count,bureau_type_other_ratio,bureau_record_per_credit_type_avg
0,100001,7,1,1,3,0.428571,4,0.571429,0,0.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,7.0
1,100002,8,2,1,2,0.250000,6,0.750000,0,0.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,4.0
2,100003,4,2,1,1,0.250000,3,0.750000,0,0.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2.0
3,100004,2,1,1,0,0.000000,2,1.000000,0,0.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,2.0
4,100005,3,2,1,2,0.666667,1,0.333333,0,0.0,...,0.0,0,0.0,0,0.0,0,0.0,0,0.0,1.5


In [32]:
# Numeric bureau feature aggregation

numeric_agg_plan = {
    "AMT_CREDIT_SUM": ("bureau_credit_amount", ["sum", "mean", "max", "min", "std"]),
    "AMT_CREDIT_SUM_DEBT": ("bureau_debt_amount", ["sum", "mean", "max", "min", "std"]),
    "AMT_CREDIT_SUM_LIMIT": ("bureau_limit_amount", ["sum", "mean", "max"]),
    "AMT_ANNUITY": ("bureau_annuity_amount", ["sum", "mean", "max"]),
    "AMT_CREDIT_SUM_OVERDUE": ("bureau_overdue_amount", ["sum", "mean", "max"]),
    "AMT_CREDIT_MAX_OVERDUE": ("bureau_max_overdue_amount", ["sum", "mean", "max"]),
    "CREDIT_DAY_OVERDUE": ("bureau_overdue_days", ["sum", "mean", "max"]),
    "CNT_CREDIT_PROLONG": ("bureau_prolong", ["sum", "mean", "max"]),
    "DAYS_CREDIT": ("bureau_days_credit", ["min", "max", "mean", "std"]),
    "DAYS_CREDIT_ENDDATE": ("bureau_days_enddate", ["min", "max", "mean"]),
    "DAYS_ENDDATE_FACT": ("bureau_days_enddate_fact", ["min", "max", "mean"]),
    "DAYS_CREDIT_UPDATE": ("bureau_days_update", ["min", "max", "mean", "std"]),
    "bureau_debt_to_credit": ("bureau_debt_to_credit", ["mean", "max", "min"]),
    "bureau_overdue_to_credit": ("bureau_overdue_to_credit", ["mean", "max"]),
    "bureau_annuity_to_credit": ("bureau_annuity_to_credit", ["mean", "max"]),
    "bureau_debt_to_limit": ("bureau_debt_to_limit", ["mean", "max"]),
    "bureau_enddate_gap": ("bureau_enddate_gap", ["mean", "max", "min"]),
}

for source_col, (prefix, agg_funcs) in numeric_agg_plan.items():
    if source_col in be.columns:
        temp = (
            be.groupby("SK_ID_CURR")[source_col]
            .agg(agg_funcs)
            .reset_index()
        )
        temp.columns = ["SK_ID_CURR"] + [
            f"{prefix}_{func}" for func in agg_funcs
        ]
        bureau_features = bureau_features.merge(temp, on="SK_ID_CURR", how="left")

bureau_features["bureau_credit_history_span"] = (
    bureau_features["bureau_days_credit_max"] - bureau_features["bureau_days_credit_min"]
)

bureau_features["bureau_total_debt_to_credit_ratio"] = (
    bureau_features["bureau_debt_amount_sum"]
    / bureau_features["bureau_credit_amount_sum"].replace(0, np.nan)
)

bureau_features["bureau_total_overdue_to_credit_ratio"] = (
    bureau_features["bureau_overdue_amount_sum"]
    / bureau_features["bureau_credit_amount_sum"].replace(0, np.nan)
)

bureau_features["bureau_total_annuity_to_credit_ratio"] = (
    bureau_features["bureau_annuity_amount_sum"]
    / bureau_features["bureau_credit_amount_sum"].replace(0, np.nan)
)

bureau_features.head()

,SK_ID_CURR,bureau_record_count,bureau_unique_credit_type_count,bureau_unique_currency_count,bureau_active_count,bureau_active_ratio,bureau_closed_count,bureau_closed_ratio,bureau_sold_count,bureau_sold_ratio,...,bureau_annuity_to_credit_max,bureau_debt_to_limit_mean,bureau_debt_to_limit_max,bureau_enddate_gap_mean,bureau_enddate_gap_max,bureau_enddate_gap_min,bureau_credit_history_span,bureau_total_debt_to_credit_ratio,bureau_total_overdue_to_credit_ratio,bureau_total_annuity_to_credit_ratio
0,100001,7,1,1,3,0.428571,4,0.571429,0,0.0,...,0.055627,NaN,NaN,-197.0,1.0,-698.0,1523,0.410555,0.0,0.017076
1,100002,8,2,1,2,0.250000,6,0.750000,0,0.0,...,0.000000,0.0,0.0,-252.6,0.0,-1029.0,1334,0.284122,0.0,0.000000
2,100003,4,2,1,1,0.250000,3,0.750000,0,0.0,...,NaN,0.0,0.0,34.0,303.0,-201.0,1980,0.000000,0.0,0.000000
3,100004,2,1,1,0,0.000000,2,1.000000,0,0.0,...,NaN,NaN,NaN,-44.0,0.0,-88.0,918,0.000000,0.0,0.000000
4,100005,3,2,1,2,0.666667,1,0.333333,0,0.0,...,0.142879,NaN,NaN,5.0,5.0,5.0,311,0.864992,0.0,0.006485


In [33]:
# Segment-based bureau features

segment_plan = {
    "active": "bureau_is_active",
    "closed": "bureau_is_closed",
    "recent_12m": "bureau_is_recent_12m",
    "recent_24m": "bureau_is_recent_24m",
    "overdue": "bureau_has_overdue_amount",
}

segment_numeric_cols = {
    "AMT_CREDIT_SUM": "credit_amount",
    "AMT_CREDIT_SUM_DEBT": "debt_amount",
    "AMT_CREDIT_SUM_LIMIT": "limit_amount",
    "AMT_ANNUITY": "annuity_amount",
    "AMT_CREDIT_SUM_OVERDUE": "overdue_amount",
    "CREDIT_DAY_OVERDUE": "overdue_days",
}

for segment_name, flag_col in segment_plan.items():
    segment_df = be[be[flag_col] == 1]
    
    if segment_df.empty:
        continue
    
    for source_col, short_name in segment_numeric_cols.items():
        temp = (
            segment_df.groupby("SK_ID_CURR")[source_col]
            .agg(["sum", "mean", "max"])
            .reset_index()
        )
        temp.columns = ["SK_ID_CURR"] + [
            f"bureau_{segment_name}_{short_name}_{func}"
            for func in ["sum", "mean", "max"]
        ]
        bureau_features = bureau_features.merge(temp, on="SK_ID_CURR", how="left")

bureau_features["bureau_active_debt_to_credit_ratio"] = (
    bureau_features["bureau_active_debt_amount_sum"]
    / bureau_features["bureau_active_credit_amount_sum"].replace(0, np.nan)
)

bureau_features["bureau_active_debt_share"] = (
    bureau_features["bureau_active_debt_amount_sum"]
    / bureau_features["bureau_debt_amount_sum"].replace(0, np.nan)
)

bureau_features.head()

,SK_ID_CURR,bureau_record_count,bureau_unique_credit_type_count,bureau_unique_currency_count,bureau_active_count,bureau_active_ratio,bureau_closed_count,bureau_closed_ratio,bureau_sold_count,bureau_sold_ratio,...,bureau_overdue_annuity_amount_mean,bureau_overdue_annuity_amount_max,bureau_overdue_overdue_amount_sum,bureau_overdue_overdue_amount_mean,bureau_overdue_overdue_amount_max,bureau_overdue_overdue_days_sum,bureau_overdue_overdue_days_mean,bureau_overdue_overdue_days_max,bureau_active_debt_to_credit_ratio,bureau_active_debt_share
0,100001,7,1,1,3,0.428571,4,0.571429,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.674966,1.0
1,100002,8,2,1,2,0.250000,6,0.750000,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.509931,1.0
2,100003,4,2,1,1,0.250000,3,0.750000,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.000000,NaN
3,100004,2,1,1,0,0.000000,2,1.000000,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,3,2,1,2,0.666667,1,0.333333,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.949522,1.0


In [34]:
# Aggregate bureau balance monthly-risk features to applicant level

bureau_balance_feature_cols = [
    col for col in be.columns
    if col.startswith("bureau_bal_")
]

bb_count_cols = [
    col for col in bureau_balance_feature_cols
    if col.endswith("_count")
]

bb_ratio_cols = [
    col for col in bureau_balance_feature_cols
    if col.endswith("_ratio") or "trend" in col
]

bb_other_cols = [
    col for col in [
        "bureau_bal_month_count",
        "bureau_bal_month_min",
        "bureau_bal_month_max",
        "bureau_bal_month_mean",
        "bureau_bal_month_span",
        "bureau_bal_unique_status_count",
        "bureau_bal_max_late_status",
    ]
    if col in be.columns
]

bb_agg_plan = {}

for col in bb_count_cols:
    bb_agg_plan[col] = ["sum", "mean", "max"]

for col in bb_ratio_cols:
    bb_agg_plan[col] = ["mean", "max", "min"]

for col in bb_other_cols:
    if col not in bb_agg_plan:
        bb_agg_plan[col] = ["mean", "max", "min"]

for source_col, agg_funcs in bb_agg_plan.items():
    temp = (
        be.groupby("SK_ID_CURR")[source_col]
        .agg(agg_funcs)
        .reset_index()
    )
    temp.columns = ["SK_ID_CURR"] + [
        f"{source_col}_{func}" for func in agg_funcs
    ]
    bureau_features = bureau_features.merge(temp, on="SK_ID_CURR", how="left")

bureau_features["bureau_without_balance_count"] = (
    bureau_features["bureau_record_count"]
    - bureau_features["bureau_with_balance_count"]
)

bureau_features["bureau_without_balance_ratio"] = (
    bureau_features["bureau_without_balance_count"]
    / bureau_features["bureau_record_count"].replace(0, np.nan)
)

bureau_features.head()

,SK_ID_CURR,bureau_record_count,bureau_unique_credit_type_count,bureau_unique_currency_count,bureau_active_count,bureau_active_ratio,bureau_closed_count,bureau_closed_ratio,bureau_sold_count,bureau_sold_ratio,...,bureau_bal_month_mean_max,bureau_bal_month_mean_min,bureau_bal_month_span_mean,bureau_bal_month_span_max,bureau_bal_month_span_min,bureau_bal_max_late_status_mean,bureau_bal_max_late_status_max,bureau_bal_max_late_status_min,bureau_without_balance_count,bureau_without_balance_ratio
0,100001,7,1,1,3,0.428571,4,0.571429,0,0.0,...,-0.5,-25.5,23.571429,51.0,1.0,0.142857,1.0,0.0,0,0.0
1,100002,8,2,1,2,0.250000,6,0.750000,0,0.0,...,-1.5,-39.5,12.750000,21.0,3.0,0.750000,1.0,0.0,0,0.0
2,100003,4,2,1,1,0.250000,3,0.750000,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4,1.0
3,100004,2,1,1,0,0.000000,2,1.000000,0,0.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2,1.0
4,100005,3,2,1,2,0.666667,1,0.333333,0,0.0,...,-1.0,-6.0,6.000000,12.0,2.0,0.000000,0.0,0.0,0,0.0


In [35]:
# Latest bureau credit snapshot features

latest_bureau = (
    be.sort_values(["SK_ID_CURR", "DAYS_CREDIT"], ascending=[True, False])
    .drop_duplicates("SK_ID_CURR")
    .reset_index(drop=True)
)

latest_features = latest_bureau[["SK_ID_CURR"]].copy()

latest_feature_map = {
    "bureau_is_active": "bureau_latest_credit_active_flag",
    "bureau_is_closed": "bureau_latest_credit_closed_flag",
    "AMT_CREDIT_SUM": "bureau_latest_credit_amount",
    "AMT_CREDIT_SUM_DEBT": "bureau_latest_debt_amount",
    "AMT_CREDIT_SUM_LIMIT": "bureau_latest_limit_amount",
    "AMT_ANNUITY": "bureau_latest_annuity_amount",
    "CREDIT_DAY_OVERDUE": "bureau_latest_overdue_days",
    "AMT_CREDIT_SUM_OVERDUE": "bureau_latest_overdue_amount",
    "bureau_debt_to_credit": "bureau_latest_debt_to_credit",
    "bureau_overdue_to_credit": "bureau_latest_overdue_to_credit",
    "DAYS_CREDIT": "bureau_latest_days_credit",
    "DAYS_CREDIT_ENDDATE": "bureau_latest_days_enddate",
    "DAYS_CREDIT_UPDATE": "bureau_latest_days_update",
}

for source_col, new_col in latest_feature_map.items():
    if source_col in latest_bureau.columns:
        latest_features[new_col] = latest_bureau[source_col]

for feature_name in selected_credit_types.keys():
    source_col = f"bureau_type_{feature_name}"
    latest_features[f"bureau_latest_type_{feature_name}_flag"] = latest_bureau[source_col]

if "bureau_bal_late_ratio" in latest_bureau.columns:
    latest_features["bureau_latest_bal_late_ratio"] = latest_bureau["bureau_bal_late_ratio"]

if "bureau_bal_severe_late_ratio" in latest_bureau.columns:
    latest_features["bureau_latest_bal_severe_late_ratio"] = latest_bureau["bureau_bal_severe_late_ratio"]

if "bureau_bal_max_late_status" in latest_bureau.columns:
    latest_features["bureau_latest_bal_max_late_status"] = latest_bureau["bureau_bal_max_late_status"]

bureau_features = bureau_features.merge(
    latest_features,
    on="SK_ID_CURR",
    how="left"
)

bureau_features.head()

,SK_ID_CURR,bureau_record_count,bureau_unique_credit_type_count,bureau_unique_currency_count,bureau_active_count,bureau_active_ratio,bureau_closed_count,bureau_closed_ratio,bureau_sold_count,bureau_sold_ratio,...,bureau_latest_type_consumer_credit_flag,bureau_latest_type_credit_card_flag,bureau_latest_type_car_loan_flag,bureau_latest_type_mortgage_flag,bureau_latest_type_microloan_flag,bureau_latest_type_business_loan_flag,bureau_latest_type_working_capital_loan_flag,bureau_latest_bal_late_ratio,bureau_latest_bal_severe_late_ratio,bureau_latest_bal_max_late_status
0,100001,7,1,1,3,0.428571,4,0.571429,0,0.0,...,1,0,0,0,0,0,0,0.0,0.0,0.0
1,100002,8,2,1,2,0.250000,6,0.750000,0,0.0,...,0,1,0,0,0,0,0,0.0,0.0,0.0
2,100003,4,2,1,1,0.250000,3,0.750000,0,0.0,...,0,1,0,0,0,0,0,NaN,NaN,NaN
3,100004,2,1,1,0,0.000000,2,1.000000,0,0.0,...,1,0,0,0,0,0,0,NaN,NaN,NaN
4,100005,3,2,1,2,0.666667,1,0.333333,0,0.0,...,1,0,0,0,0,0,0,0.0,0.0,0.0


In [36]:
# Final bureau feature block information and save

bureau_features = bureau_features.replace([np.inf, -np.inf], np.nan)

bureau_feature_block_info = pd.DataFrame([
    {
        "feature_block": "bureau_features",
        "rows": bureau_features.shape[0],
        "columns": bureau_features.shape[1],
        "unique_SK_ID_CURR": bureau_features["SK_ID_CURR"].nunique(),
        "duplicate_SK_ID_CURR": bureau_features.duplicated("SK_ID_CURR").sum(),
        "missing_cells": bureau_features.isna().sum().sum(),
    }
])

bureau_features.to_csv(
    OUTPUT_DIR / "bureau_features.csv",
    index=False
)

bureau_feature_block_info.to_csv(
    OUTPUT_DIR / "bureau_feature_block_info.csv",
    index=False
)

bureau_feature_block_info

,feature_block,rows,columns,unique_SK_ID_CURR,duplicate_SK_ID_CURR,missing_cells
0,bureau_features,305811,344,305811,0,32332925


In [37]:
bureau_features.head()

,SK_ID_CURR,bureau_record_count,bureau_unique_credit_type_count,bureau_unique_currency_count,bureau_active_count,bureau_active_ratio,bureau_closed_count,bureau_closed_ratio,bureau_sold_count,bureau_sold_ratio,...,bureau_latest_type_consumer_credit_flag,bureau_latest_type_credit_card_flag,bureau_latest_type_car_loan_flag,bureau_latest_type_mortgage_flag,bureau_latest_type_microloan_flag,bureau_latest_type_business_loan_flag,bureau_latest_type_working_capital_loan_flag,bureau_latest_bal_late_ratio,bureau_latest_bal_severe_late_ratio,bureau_latest_bal_max_late_status
0,100001,7,1,1,3,0.428571,4,0.571429,0,0.0,...,1,0,0,0,0,0,0,0.0,0.0,0.0
1,100002,8,2,1,2,0.250000,6,0.750000,0,0.0,...,0,1,0,0,0,0,0,0.0,0.0,0.0
2,100003,4,2,1,1,0.250000,3,0.750000,0,0.0,...,0,1,0,0,0,0,0,NaN,NaN,NaN
3,100004,2,1,1,0,0.000000,2,1.000000,0,0.0,...,1,0,0,0,0,0,0,NaN,NaN,NaN
4,100005,3,2,1,2,0.666667,1,0.333333,0,0.0,...,1,0,0,0,0,0,0,0.0,0.0,0.0


### Bureau Feature Block Decision

The full `bureau_features` block is kept at this stage.

Although the block contains many features, they are generated from meaningful external-credit behavior patterns such as credit history size, active/closed credit status, debt burden, overdue risk, credit recency, credit type pattern, bureau-balance monthly risk, and latest bureau-credit snapshot.

No feature is removed during data engineering.  
Feature filtering will be performed later in the feature engineering/modeling stage using correlation analysis, Cramér’s V for categorical relationships, missingness, low-variance checks, feature importance, and cross-validation performance.

Final decision: keep the full `bureau_features` candidate block for now.

### 2.1.6–2.1.7 Bureau Feature Block Quality Check and Save

In this step, we check the final `bureau_features` block and save it for later integration.

The main purpose is to confirm that the bureau feature block is applicant-level:

| Check                    | Meaning                                      |
| ------------------------ | -------------------------------------------- |
| row count                | how many applicants have bureau history      |
| column count             | how many bureau features were created        |
| unique `SK_ID_CURR`      | applicant-level uniqueness                   |
| duplicate `SK_ID_CURR`   | whether row explosion happened               |
| missing feature snapshot | which bureau features contain missing values |

The full bureau feature block is kept as a candidate feature set.
Feature filtering will be handled later during feature engineering and modeling.


In [38]:
# Final bureau feature block quality check and save

bureau_feature_block_info = pd.DataFrame([{
    "feature_block": "bureau_features",
    "rows": bureau_features.shape[0],
    "columns": bureau_features.shape[1],
    "unique_SK_ID_CURR": bureau_features["SK_ID_CURR"].nunique(),
    "duplicate_SK_ID_CURR": bureau_features.duplicated("SK_ID_CURR").sum(),
    "total_missing_cells": bureau_features.isna().sum().sum(),
    "missing_columns_count": (bureau_features.isna().sum() > 0).sum(),
}])

bureau_feature_missing_info = (
    bureau_features
    .isna()
    .mean()
    .mul(100)
    .reset_index()
)

bureau_feature_missing_info.columns = ["feature_name", "missing_percent"]

bureau_feature_missing_info = (
    bureau_feature_missing_info
    .query("missing_percent > 0")
    .sort_values("missing_percent", ascending=False)
    .reset_index(drop=True)
)

bureau_features.to_csv(OUTPUT_DIR / "bureau_features.csv", index=False)
bureau_feature_block_info.to_csv(OUTPUT_DIR / "bureau_feature_block_info.csv", index=False)
bureau_feature_missing_info.to_csv(OUTPUT_DIR / "bureau_feature_missing_info.csv", index=False)

bureau_feature_block_info

,feature_block,rows,columns,unique_SK_ID_CURR,duplicate_SK_ID_CURR,total_missing_cells,missing_columns_count
0,bureau_features,305811,344,305811,0,32332925,239


In [39]:
bureau_feature_missing_info.head(30)

,feature_name,missing_percent
0,bureau_overdue_annuity_amount_max,99.657959
1,bureau_overdue_annuity_amount_mean,99.657959
2,bureau_overdue_limit_amount_max,99.251825
3,bureau_overdue_limit_amount_mean,99.251825
4,bureau_overdue_debt_amount_max,98.884278
5,bureau_overdue_debt_amount_mean,98.884278
6,bureau_overdue_credit_amount_mean,98.757075
7,bureau_overdue_debt_amount_sum,98.757075
8,bureau_overdue_credit_amount_max,98.757075
9,bureau_overdue_credit_amount_sum,98.757075


### 2.1 Final Decision: Bureau and Bureau Balance Feature Block

| Area                 | Observation                                                                                                                                                                                                | Decision                                                                                                                              |
| -------------------- | ---------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------- | ------------------------------------------------------------------------------------------------------------------------------------- |
| Relationship path    | `bureau_balance` has only `SK_ID_BUREAU`, while `bureau` has both `SK_ID_BUREAU` and `SK_ID_CURR`.                                                                                                         | `bureau_balance` was first aggregated by `SK_ID_BUREAU`, then merged with `bureau`.                                                   |
| Bureau balance grain | Raw `bureau_balance` contains many monthly rows for each `SK_ID_BUREAU`.                                                                                                                                   | It was converted into one row per `SK_ID_BUREAU` before merging.                                                                      |
| Bureau grain         | Raw `bureau` contains multiple external credit records per applicant.                                                                                                                                      | After enrichment, `bureau_enriched` was aggregated by `SK_ID_CURR`.                                                                   |
| Final block          | `bureau_features` is now applicant-level.                                                                                                                                                                  | It is safe for later merging with application train/test by `SK_ID_CURR`.                                                             |
| Feature type         | Features represent external credit history, active/closed credit behavior, debt burden, overdue risk, credit recency, credit type pattern, bureau-balance monthly risk, and latest bureau-credit snapshot. | The full feature block is kept as a candidate external-credit feature set.                                                            |
| Feature count        | The block contains many generated features.                                                                                                                                                                | No feature is removed in this data-engineering notebook. Feature filtering will be handled later during feature engineering/modeling. |
| Missing values       | Some missing values are expected, especially where applicants or bureau records do not have bureau-balance history.                                                                                        | No imputation is done here. Missingness will be reviewed later.                                                                       |
| Row safety           | The final block preserves one row per `SK_ID_CURR`.                                                                                                                                                        | This prevents row explosion during final integration.                                                                                 |

**Final decision:** The `bureau` and `bureau_balance` tables were successfully converted into an applicant-level candidate feature block named `bureau_features`. This block will be saved and used later during final integration. No raw bureau-related table will be merged directly with the application base.


### 2.2.1 Previous Application Relationship Information

In this section, we collect basic relationship information from `previous_application`.

This table contains previous Home Credit applications for current applicants.  
One applicant can have multiple previous applications, so repeated `SK_ID_CURR` is expected.

We only collect simple structural information here: row count, unique ID count, missing key values, duplicate previous application IDs, and how many current applicants have previous application history.

In [40]:
# Previous application relationship information

previous_application = data["previous_application"]

train_ids = set(data["application_train"]["SK_ID_CURR"])
test_ids = set(data["application_test"]["SK_ID_CURR"])
application_ids = train_ids | test_ids

previous_curr_ids = set(previous_application["SK_ID_CURR"].dropna().unique())

previous_relation_info = pd.DataFrame([{
    "table_name": "previous_application",
    "rows": previous_application.shape[0],
    "columns": previous_application.shape[1],
    "unique_SK_ID_CURR": previous_application["SK_ID_CURR"].nunique(),
    "unique_SK_ID_PREV": previous_application["SK_ID_PREV"].nunique(),
    "missing_SK_ID_CURR": previous_application["SK_ID_CURR"].isna().sum(),
    "missing_SK_ID_PREV": previous_application["SK_ID_PREV"].isna().sum(),
    "duplicate_SK_ID_PREV": previous_application.duplicated("SK_ID_PREV").sum(),
    "repeated_SK_ID_CURR_rows": previous_application.duplicated("SK_ID_CURR").sum(),
}])

previous_application_coverage_info = pd.DataFrame([{
    "application_all_applicants": len(application_ids),
    "previous_application_applicants": len(previous_curr_ids),
    "application_applicants_with_previous_history": len(application_ids & previous_curr_ids),
    "application_applicants_without_previous_history": len(application_ids - previous_curr_ids),
    "previous_applicants_not_in_application": len(previous_curr_ids - application_ids),
    "train_applicants_with_previous_history": len(train_ids & previous_curr_ids),
    "test_applicants_with_previous_history": len(test_ids & previous_curr_ids),
}])

previous_rows_per_applicant_info = (
    previous_application
    .groupby("SK_ID_CURR")
    .size()
    .describe()
    .reset_index()
)

previous_rows_per_applicant_info.columns = ["metric", "previous_application_count"]

previous_relation_info.to_csv(
    OUTPUT_DIR / "previous_relation_info.csv",
    index=False
)

previous_application_coverage_info.to_csv(
    OUTPUT_DIR / "previous_application_coverage_info.csv",
    index=False
)

previous_rows_per_applicant_info.to_csv(
    OUTPUT_DIR / "previous_rows_per_applicant_info.csv",
    index=False
)

previous_relation_info

,table_name,rows,columns,unique_SK_ID_CURR,unique_SK_ID_PREV,missing_SK_ID_CURR,missing_SK_ID_PREV,duplicate_SK_ID_PREV,repeated_SK_ID_CURR_rows
0,previous_application,1670214,37,338857,1670214,0,0,0,1331357


In [41]:
previous_application_coverage_info

,application_all_applicants,previous_application_applicants,application_applicants_with_previous_history,application_applicants_without_previous_history,previous_applicants_not_in_application,train_applicants_with_previous_history,test_applicants_with_previous_history
0,356255,338857,338857,17398,0,291057,47800


In [42]:
previous_rows_per_applicant_info

,metric,previous_application_count
0,count,338857.000000
1,mean,4.928964
2,std,4.220716
3,min,1.000000
4,25%,2.000000
5,50%,4.000000
6,75%,7.000000
7,max,77.000000


### 2.2.1 Summary: Previous Application Relationship

| Area                       | Observation                                                                                                |
| -------------------------- | ---------------------------------------------------------------------------------------------------------- |
| Table size                 | `previous_application` has 1,670,214 rows and 37 columns.                                                  |
| Applicant coverage         | 338,857 applicants have previous Home Credit application history.                                          |
| Previous application ID    | `SK_ID_PREV` is unique, so each row represents one previous application record.                            |
| Missing keys               | `SK_ID_CURR` and `SK_ID_PREV` have 0 missing values.                                                       |
| Repeated applicants        | `SK_ID_CURR` is repeated, which is expected because one applicant can have multiple previous applications. |
| Coverage gap               | 17,398 current applicants do not have previous application history.                                        |
| Applications per applicant | On average, each applicant has about 4.93 previous applications; the maximum is 77.                        |

**Decision:** `previous_application` is a clean one-to-many history table. It should not be merged directly with application data. It will be summarized into one applicant-level feature block using `SK_ID_CURR`.


### 2.2.2 Previous Application Category and Numeric Information

In this section, we inspect useful categorical, numeric, time, rate, and flag columns from `previous_application`.

No aggregation or feature creation is done here.  
The goal is to understand which columns can provide useful signals for the later applicant-level feature block.

In [43]:
# Previous application categorical information

previous_application = data["previous_application"]

previous_category_cols = previous_application.select_dtypes(include="object").columns.tolist()

previous_category_info = []

for col in previous_category_cols:
    previous_category_info.append({
        "column_name": col,
        "unique_values": previous_application[col].nunique(dropna=True),
        "missing_values": previous_application[col].isna().sum(),
        "missing_percent": round(previous_application[col].isna().mean() * 100, 4),
        "top_value": previous_application[col].mode(dropna=True)[0] if not previous_application[col].mode(dropna=True).empty else "None",
        "top_value_count": previous_application[col].value_counts(dropna=True).iloc[0] if previous_application[col].nunique(dropna=True) > 0 else 0,
    })

previous_category_info = pd.DataFrame(previous_category_info)

previous_category_info.to_csv(
    OUTPUT_DIR / "previous_category_info.csv",
    index=False
)

previous_category_info

,column_name,unique_values,missing_values,missing_percent,top_value,top_value_count
0,NAME_CONTRACT_TYPE,4,0,0.0000,Cash loans,747553
1,WEEKDAY_APPR_PROCESS_START,7,0,0.0000,TUESDAY,255118
2,FLAG_LAST_APPL_PER_CONTRACT,2,0,0.0000,Y,1661739
3,NAME_CASH_LOAN_PURPOSE,25,0,0.0000,XAP,922661
4,NAME_CONTRACT_STATUS,4,0,0.0000,Approved,1036781
5,NAME_PAYMENT_TYPE,4,0,0.0000,Cash through the bank,1033552
6,CODE_REJECT_REASON,9,0,0.0000,XAP,1353093
7,NAME_TYPE_SUITE,7,820405,49.1198,Unaccompanied,508970
8,NAME_CLIENT_TYPE,4,0,0.0000,Repeater,1231261
9,NAME_GOODS_CATEGORY,28,0,0.0000,XNA,950809


In [44]:
# Key categorical value counts

key_category_cols = [
    "NAME_CONTRACT_STATUS",
    "NAME_CONTRACT_TYPE",
    "NAME_CLIENT_TYPE",
    "NAME_PORTFOLIO",
    "CHANNEL_TYPE",
    "NAME_SELLER_INDUSTRY",
    "NAME_YIELD_GROUP",
    "PRODUCT_COMBINATION",
]

key_category_cols = [
    col for col in key_category_cols
    if col in previous_application.columns
]

category_count_records = []

for col in key_category_cols:
    value_counts = previous_application[col].value_counts(dropna=False)
    
    for value, count in value_counts.items():
        category_count_records.append({
            "column_name": col,
            "category_value": value,
            "count": count,
            "percent": round(count / len(previous_application) * 100, 4),
        })

previous_key_category_value_counts = pd.DataFrame(category_count_records)

previous_key_category_value_counts.to_csv(
    OUTPUT_DIR / "previous_key_category_value_counts.csv",
    index=False
)

previous_key_category_value_counts

,column_name,category_value,count,percent
0,NAME_CONTRACT_STATUS,Approved,1036781,62.0747
1,NAME_CONTRACT_STATUS,Canceled,316319,18.9388
2,NAME_CONTRACT_STATUS,Refused,290678,17.4036
3,NAME_CONTRACT_STATUS,Unused offer,26436,1.5828
4,NAME_CONTRACT_TYPE,Cash loans,747553,44.7579
5,NAME_CONTRACT_TYPE,Consumer loans,729151,43.6561
6,NAME_CONTRACT_TYPE,Revolving loans,193164,11.5652
7,NAME_CONTRACT_TYPE,XNA,346,0.0207
8,NAME_CLIENT_TYPE,Repeater,1231261,73.7188
9,NAME_CLIENT_TYPE,New,301363,18.0434


In [45]:
# Previous application amount column information

previous_amount_cols = [
    "AMT_APPLICATION",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "AMT_GOODS_PRICE",
    "AMT_DOWN_PAYMENT",
]

previous_amount_cols = [
    col for col in previous_amount_cols
    if col in previous_application.columns
]

previous_amount_info = (
    previous_application[previous_amount_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column_name"})
)

previous_amount_info["missing_values"] = [
    previous_application[col].isna().sum() for col in previous_amount_cols
]

previous_amount_info["missing_percent"] = [
    round(previous_application[col].isna().mean() * 100, 4) for col in previous_amount_cols
]

previous_amount_info["zero_values"] = [
    (previous_application[col] == 0).sum() for col in previous_amount_cols
]

previous_amount_info.to_csv(
    OUTPUT_DIR / "previous_amount_info.csv",
    index=False
)

previous_amount_info

,column_name,count,mean,std,min,25%,50%,75%,max,missing_values,missing_percent,zero_values
0,AMT_APPLICATION,1670214.0,175233.860360,292779.762386,0.0,18720.00,71046.0,180360.00,6905160.000,0,0.0000,392402
1,AMT_CREDIT,1670213.0,196114.021218,318574.616547,0.0,24160.50,80541.0,216418.50,6905160.000,1,0.0001,336768
2,AMT_ANNUITY,1297979.0,15955.120659,14782.137335,0.0,6321.78,11250.0,20658.42,418058.145,372235,22.2867,1637
3,AMT_GOODS_PRICE,1284699.0,227847.279283,315396.557937,0.0,50841.00,112320.0,234000.00,6905160.000,385515,23.0818,6869
4,AMT_DOWN_PAYMENT,774370.0,6697.402139,20921.495410,-0.9,0.00,1638.0,7740.00,3060045.000,895844,53.6365,369854


In [46]:
# Time, rate, count, and flag column information

previous_time_rate_flag_cols = [
    "DAYS_DECISION",
    "DAYS_FIRST_DRAWING",
    "DAYS_FIRST_DUE",
    "DAYS_LAST_DUE_1ST_VERSION",
    "DAYS_LAST_DUE",
    "DAYS_TERMINATION",
    "CNT_PAYMENT",
    "RATE_DOWN_PAYMENT",
    "RATE_INTEREST_PRIMARY",
    "RATE_INTEREST_PRIVILEGED",
    "HOUR_APPR_PROCESS_START",
    "SELLERPLACE_AREA",
    "NFLAG_LAST_APPL_IN_DAY",
    "NFLAG_INSURED_ON_APPROVAL",
]

previous_time_rate_flag_cols = [
    col for col in previous_time_rate_flag_cols
    if col in previous_application.columns
]

previous_time_rate_flag_info = (
    previous_application[previous_time_rate_flag_cols]
    .describe()
    .T
    .reset_index()
    .rename(columns={"index": "column_name"})
)

previous_time_rate_flag_info["missing_values"] = [
    previous_application[col].isna().sum() for col in previous_time_rate_flag_cols
]

previous_time_rate_flag_info["missing_percent"] = [
    round(previous_application[col].isna().mean() * 100, 4) for col in previous_time_rate_flag_cols
]

previous_time_rate_flag_info.to_csv(
    OUTPUT_DIR / "previous_time_rate_flag_info.csv",
    index=False
)

previous_time_rate_flag_info

,column_name,count,mean,std,min,25%,50%,75%,max,missing_values,missing_percent
0,DAYS_DECISION,1670214.0,-880.679668,779.099667,-2922.000000,-1300.000000,-581.000000,-280.000000,-1.0,0,0.0000
1,DAYS_FIRST_DRAWING,997149.0,342209.855039,88916.115833,-2922.000000,365243.000000,365243.000000,365243.000000,365243.0,673065,40.2981
2,DAYS_FIRST_DUE,997149.0,13826.269337,72444.869708,-2892.000000,-1628.000000,-831.000000,-411.000000,365243.0,673065,40.2981
3,DAYS_LAST_DUE_1ST_VERSION,997149.0,33767.774054,106857.034789,-2801.000000,-1242.000000,-361.000000,129.000000,365243.0,673065,40.2981
4,DAYS_LAST_DUE,997149.0,76582.403064,149647.415123,-2889.000000,-1314.000000,-537.000000,-74.000000,365243.0,673065,40.2981
5,DAYS_TERMINATION,997149.0,81992.343838,153303.516729,-2874.000000,-1270.000000,-499.000000,-44.000000,365243.0,673065,40.2981
6,CNT_PAYMENT,1297984.0,16.054082,14.567288,0.000000,6.000000,12.000000,24.000000,84.0,372230,22.2864
7,RATE_DOWN_PAYMENT,774370.0,0.079637,0.107823,-0.000015,0.000000,0.051605,0.108909,1.0,895844,53.6365
8,RATE_INTEREST_PRIMARY,5951.0,0.188357,0.087671,0.034781,0.160716,0.189122,0.193330,1.0,1664263,99.6437
9,RATE_INTEREST_PRIVILEGED,5951.0,0.773503,0.100879,0.373150,0.715645,0.835095,0.852537,1.0,1664263,99.6437


### 2.2.2 Summary: Previous Application Signal Areas

| Area                  | Observation                                                                                                                 |
| --------------------- | --------------------------------------------------------------------------------------------------------------------------- |
| Outcome status        | Most previous applications are `Approved` around 62.07%, while `Canceled` and `Refused` are also common.                    |
| Contract type         | `Cash loans` and `Consumer loans` are the two dominant previous contract types; `Revolving loans` is smaller but important. |
| Client behavior       | Most records are from `Repeater` clients, so previous relationship with Home Credit is a strong signal.                     |
| Product/channel       | `POS`, `Cash`, `Cards`, `Credit and cash offices`, and `Country-wide` are major product/channel patterns.                   |
| Amount columns        | `AMT_APPLICATION`, `AMT_CREDIT`, `AMT_ANNUITY`, and `AMT_GOODS_PRICE` can create strong amount and burden features.         |
| Missing amount fields | `AMT_DOWN_PAYMENT`, `RATE_DOWN_PAYMENT`, and some rate columns have high missingness, so they should be used carefully.     |
| Time columns          | `DAYS_DECISION` has no missing values and can capture previous application recency.                                         |
| Placeholder issue     | Some previous time columns contain `365243`, so these should be treated as special placeholder values later.                |

**Decision:** `previous_application` contains strong signals about past Home Credit behavior. The next step will create row-level helper features for outcome, contract type, recency, amount ratios, payment terms, and selected process indicators before aggregating to `SK_ID_CURR`.


### 2.2.3 Previous Application Row-Level Derived Features

In this step, we create row-level helper features from `previous_application`.

These features capture previous application outcome, contract type, recency, amount ratios, amount gaps, payment term, client behavior, product/channel pattern, yield group, and process indicators.

No aggregation is done here.  
The derived table will be used in the next step to create applicant-level `previous_features`.

In [47]:
# Create row-level derived features from previous_application

import numpy as np

pa = data["previous_application"].copy()

# Replace known placeholder day values
day_cols = [col for col in pa.columns if col.startswith("DAYS_")]
pa[day_cols] = pa[day_cols].replace(365243, np.nan)

# Normalize important categorical columns
category_cols = [
    "NAME_CONTRACT_STATUS", "NAME_CONTRACT_TYPE", "NAME_CLIENT_TYPE",
    "NAME_PORTFOLIO", "CHANNEL_TYPE", "NAME_YIELD_GROUP"
]

for col in category_cols:
    if col in pa.columns:
        pa[col] = pa[col].astype(str).str.strip().str.lower()

# Outcome flags
pa["prev_is_approved"] = pa["NAME_CONTRACT_STATUS"].eq("approved").astype(int)
pa["prev_is_refused"] = pa["NAME_CONTRACT_STATUS"].eq("refused").astype(int)
pa["prev_is_canceled"] = pa["NAME_CONTRACT_STATUS"].eq("canceled").astype(int)
pa["prev_is_unused_offer"] = pa["NAME_CONTRACT_STATUS"].eq("unused offer").astype(int)

# Contract type flags
pa["prev_is_cash_loan"] = pa["NAME_CONTRACT_TYPE"].eq("cash loans").astype(int)
pa["prev_is_consumer_loan"] = pa["NAME_CONTRACT_TYPE"].eq("consumer loans").astype(int)
pa["prev_is_revolving_loan"] = pa["NAME_CONTRACT_TYPE"].eq("revolving loans").astype(int)
pa["prev_is_contract_xna"] = pa["NAME_CONTRACT_TYPE"].eq("xna").astype(int)

# Recency flags using DAYS_DECISION
pa["prev_is_recent_6m"] = (pa["DAYS_DECISION"] >= -180).astype(int)
pa["prev_is_recent_12m"] = (pa["DAYS_DECISION"] >= -365).astype(int)
pa["prev_is_recent_24m"] = (pa["DAYS_DECISION"] >= -730).astype(int)

# Safe ratio helper
def safe_ratio(numerator, denominator):
    return numerator / denominator.replace(0, np.nan)

# Amount ratio features
pa["prev_credit_to_application_ratio"] = safe_ratio(pa["AMT_CREDIT"], pa["AMT_APPLICATION"])
pa["prev_annuity_to_credit_ratio"] = safe_ratio(pa["AMT_ANNUITY"], pa["AMT_CREDIT"])
pa["prev_down_payment_ratio"] = safe_ratio(pa["AMT_DOWN_PAYMENT"], pa["AMT_GOODS_PRICE"])
pa["prev_goods_to_application_ratio"] = safe_ratio(pa["AMT_GOODS_PRICE"], pa["AMT_APPLICATION"])
pa["prev_credit_to_goods_ratio"] = safe_ratio(pa["AMT_CREDIT"], pa["AMT_GOODS_PRICE"])

# Amount gap features
pa["prev_application_credit_diff"] = pa["AMT_APPLICATION"] - pa["AMT_CREDIT"]
pa["prev_credit_goods_diff"] = pa["AMT_CREDIT"] - pa["AMT_GOODS_PRICE"]
pa["prev_application_goods_diff"] = pa["AMT_APPLICATION"] - pa["AMT_GOODS_PRICE"]

# Payment term flags
pa["prev_is_short_term"] = (pa["CNT_PAYMENT"] <= 12).astype(int)
pa["prev_is_medium_term"] = ((pa["CNT_PAYMENT"] > 12) & (pa["CNT_PAYMENT"] <= 24)).astype(int)
pa["prev_is_long_term"] = (pa["CNT_PAYMENT"] > 24).astype(int)

# Client type flags
pa["prev_is_repeater_client"] = pa["NAME_CLIENT_TYPE"].eq("repeater").astype(int)
pa["prev_is_new_client"] = pa["NAME_CLIENT_TYPE"].eq("new").astype(int)
pa["prev_is_refreshed_client"] = pa["NAME_CLIENT_TYPE"].eq("refreshed").astype(int)

# Portfolio flags
pa["prev_portfolio_cash"] = pa["NAME_PORTFOLIO"].eq("cash").astype(int)
pa["prev_portfolio_cards"] = pa["NAME_PORTFOLIO"].eq("cards").astype(int)
pa["prev_portfolio_pos"] = pa["NAME_PORTFOLIO"].eq("pos").astype(int)
pa["prev_portfolio_cars"] = pa["NAME_PORTFOLIO"].eq("cars").astype(int)
pa["prev_portfolio_xna"] = pa["NAME_PORTFOLIO"].eq("xna").astype(int)

# Channel flags
pa["prev_channel_credit_cash_office"] = pa["CHANNEL_TYPE"].eq("credit and cash offices").astype(int)
pa["prev_channel_country_wide"] = pa["CHANNEL_TYPE"].eq("country-wide").astype(int)
pa["prev_channel_contact_center"] = pa["CHANNEL_TYPE"].eq("contact center").astype(int)
pa["prev_channel_ap_cash_loan"] = pa["CHANNEL_TYPE"].eq("ap+ (cash loan)").astype(int)
pa["prev_channel_regional_local"] = pa["CHANNEL_TYPE"].eq("regional / local").astype(int)
pa["prev_channel_corporate_sales"] = pa["CHANNEL_TYPE"].eq("channel of corporate sales").astype(int)

# Yield group flags
pa["prev_yield_high"] = pa["NAME_YIELD_GROUP"].eq("high").astype(int)
pa["prev_yield_middle"] = pa["NAME_YIELD_GROUP"].eq("middle").astype(int)
pa["prev_yield_low_normal"] = pa["NAME_YIELD_GROUP"].eq("low_normal").astype(int)
pa["prev_yield_low_action"] = pa["NAME_YIELD_GROUP"].eq("low_action").astype(int)
pa["prev_yield_xna"] = pa["NAME_YIELD_GROUP"].eq("xna").astype(int)

# Process and approval-related flags
pa["prev_last_appl_in_day"] = pa["NFLAG_LAST_APPL_IN_DAY"]
pa["prev_insured_on_approval"] = pa["NFLAG_INSURED_ON_APPROVAL"]
pa["prev_has_down_payment"] = (pa["AMT_DOWN_PAYMENT"].fillna(0) > 0).astype(int)
pa["prev_has_interest_rate"] = pa["RATE_INTEREST_PRIMARY"].notna().astype(int)

# Derived feature info
previous_derived_cols = [
    col for col in pa.columns
    if col.startswith("prev_")
]

previous_derived_feature_info = pd.DataFrame([{
    "derived_table": "previous_application_derived",
    "rows": pa.shape[0],
    "columns": pa.shape[1],
    "derived_feature_count": len(previous_derived_cols),
    "unique_SK_ID_CURR": pa["SK_ID_CURR"].nunique(),
    "unique_SK_ID_PREV": pa["SK_ID_PREV"].nunique(),
}])

previous_derived_feature_info.to_csv(
    OUTPUT_DIR / "previous_derived_feature_info.csv",
    index=False
)

previous_derived_feature_info

,derived_table,rows,columns,derived_feature_count,unique_SK_ID_CURR,unique_SK_ID_PREV
0,previous_application_derived,1670214,82,45,338857,1670214


In [48]:
pa[["SK_ID_CURR", "SK_ID_PREV"] + previous_derived_cols[:20]].head()

,SK_ID_CURR,SK_ID_PREV,prev_is_approved,prev_is_refused,prev_is_canceled,prev_is_unused_offer,prev_is_cash_loan,prev_is_consumer_loan,prev_is_revolving_loan,prev_is_contract_xna,...,prev_is_recent_24m,prev_credit_to_application_ratio,prev_annuity_to_credit_ratio,prev_down_payment_ratio,prev_goods_to_application_ratio,prev_credit_to_goods_ratio,prev_application_credit_diff,prev_credit_goods_diff,prev_application_goods_diff,prev_is_short_term
0,271877,2030495,1,0,0,0,0,1,0,0,...,1,1.00000,0.100929,0.0,1.0,1.00000,0.0,0.0,0.0,1
1,108129,2802425,1,0,0,0,1,0,0,0,...,1,1.11880,0.037060,NaN,1.0,1.11880,-72171.0,72171.0,0.0,0
2,122040,2523466,1,0,0,0,1,0,0,0,...,1,1.21284,0.110380,NaN,1.0,1.21284,-23944.5,23944.5,0.0,1
3,176158,2819243,1,0,0,0,1,0,0,0,...,1,1.04620,0.099920,NaN,1.0,1.04620,-20790.0,20790.0,0.0,1
4,202054,1784265,0,1,0,0,1,0,0,0,...,0,1.19720,0.079010,NaN,1.0,1.19720,-66555.0,66555.0,0.0,0


### 2.2.4 Create Applicant-Level Previous Application Feature Block

In this step, we aggregate the row-level derived previous application table into applicant-level features.

Current structure:

```text
pa
many rows per SK_ID_CURR

In [49]:

previous_features = (
    pa.groupby("SK_ID_CURR")
    .agg(
        prev_application_count=("SK_ID_PREV", "count"),
        prev_unique_contract_type_count=("NAME_CONTRACT_TYPE", "nunique"),
        prev_unique_contract_status_count=("NAME_CONTRACT_STATUS", "nunique"),
        prev_unique_client_type_count=("NAME_CLIENT_TYPE", "nunique"),
        prev_unique_portfolio_count=("NAME_PORTFOLIO", "nunique"),
        prev_unique_channel_count=("CHANNEL_TYPE", "nunique"),
        prev_unique_product_combination_count=("PRODUCT_COMBINATION", "nunique"),
    )
    .reset_index()
)

prev_flag_cols = [
    "prev_is_approved", "prev_is_refused", "prev_is_canceled", "prev_is_unused_offer",
    "prev_is_cash_loan", "prev_is_consumer_loan", "prev_is_revolving_loan", "prev_is_contract_xna",
    "prev_is_recent_6m", "prev_is_recent_12m", "prev_is_recent_24m",
    "prev_is_short_term", "prev_is_medium_term", "prev_is_long_term",
    "prev_is_repeater_client", "prev_is_new_client", "prev_is_refreshed_client",
    "prev_portfolio_cash", "prev_portfolio_cards", "prev_portfolio_pos", "prev_portfolio_cars", "prev_portfolio_xna",
    "prev_channel_credit_cash_office", "prev_channel_country_wide", "prev_channel_contact_center",
    "prev_channel_ap_cash_loan", "prev_channel_regional_local", "prev_channel_corporate_sales",
    "prev_yield_high", "prev_yield_middle", "prev_yield_low_normal", "prev_yield_low_action", "prev_yield_xna",
    "prev_last_appl_in_day", "prev_insured_on_approval", "prev_has_down_payment", "prev_has_interest_rate",
]

prev_flag_cols = [col for col in prev_flag_cols if col in pa.columns]

for col in prev_flag_cols:
    temp = (
        pa.groupby("SK_ID_CURR")[col]
        .agg(["sum", "mean"])
        .reset_index()
    )
    clean_name = col.replace("prev_is_", "prev_").replace("prev_", "")
    temp.columns = ["SK_ID_CURR", f"prev_{clean_name}_count", f"prev_{clean_name}_ratio"]
    previous_features = previous_features.merge(temp, on="SK_ID_CURR", how="left")

previous_features["prev_refused_to_approved_ratio"] = (
    previous_features["prev_refused_count"]
    / previous_features["prev_approved_count"].replace(0, np.nan)
)

previous_features.head()

,SK_ID_CURR,prev_application_count,prev_unique_contract_type_count,prev_unique_contract_status_count,prev_unique_client_type_count,prev_unique_portfolio_count,prev_unique_channel_count,prev_unique_product_combination_count,prev_approved_count,prev_approved_ratio,...,prev_yield_xna_ratio,prev_last_appl_in_day_count,prev_last_appl_in_day_ratio,prev_insured_on_approval_count,prev_insured_on_approval_ratio,prev_has_down_payment_count,prev_has_down_payment_ratio,prev_has_interest_rate_count,prev_has_interest_rate_ratio,prev_refused_to_approved_ratio
0,100001,1,1,1,1,1,1,1,1,1.0,...,0.0,1,1.0,0.0,0.000000,1,1.000000,0,0.0,0.0
1,100002,1,1,1,1,1,1,1,1,1.0,...,0.0,1,1.0,0.0,0.000000,0,0.000000,0,0.0,0.0
2,100003,3,2,1,2,2,3,3,3,1.0,...,0.0,3,1.0,2.0,0.666667,1,0.333333,0,0.0,0.0
3,100004,1,1,1,1,1,1,1,1,1.0,...,0.0,1,1.0,0.0,0.000000,1,1.000000,0,0.0,0.0
4,100005,2,2,2,2,2,2,2,1,0.5,...,0.5,2,1.0,0.0,0.000000,1,0.500000,0,0.0,0.0


In [50]:
# Segment-based previous application features

prev_segment_plan = {
    "approved": "prev_is_approved",
    "refused": "prev_is_refused",
    "canceled": "prev_is_canceled",
    "unused_offer": "prev_is_unused_offer",
}

prev_segment_numeric_cols = {
    "AMT_APPLICATION": "application_amount",
    "AMT_CREDIT": "credit_amount",
    "AMT_ANNUITY": "annuity_amount",
    "AMT_GOODS_PRICE": "goods_price",
    "AMT_DOWN_PAYMENT": "down_payment",
    "CNT_PAYMENT": "cnt_payment",
    "prev_credit_to_application_ratio": "credit_to_application_ratio",
    "prev_annuity_to_credit_ratio": "annuity_to_credit_ratio",
    "prev_application_credit_diff": "application_credit_diff",
}

for segment_name, flag_col in prev_segment_plan.items():
    segment_df = pa[pa[flag_col] == 1]
    
    if segment_df.empty:
        continue
    
    segment_count = (
        segment_df.groupby("SK_ID_CURR")["SK_ID_PREV"]
        .count()
        .reset_index()
    )
    segment_count.columns = ["SK_ID_CURR", f"prev_{segment_name}_application_count"]
    previous_features = previous_features.merge(segment_count, on="SK_ID_CURR", how="left")
    
    for source_col, short_name in prev_segment_numeric_cols.items():
        if source_col in segment_df.columns:
            temp = (
                segment_df.groupby("SK_ID_CURR")[source_col]
                .agg(["sum", "mean", "max"])
                .reset_index()
            )
            temp.columns = ["SK_ID_CURR"] + [
                f"prev_{segment_name}_{short_name}_{func}"
                for func in ["sum", "mean", "max"]
            ]
            previous_features = previous_features.merge(temp, on="SK_ID_CURR", how="left")
    
    for recent_col in ["prev_is_recent_12m", "prev_is_recent_24m"]:
        if recent_col in segment_df.columns:
            temp = (
                segment_df.groupby("SK_ID_CURR")[recent_col]
                .sum()
                .reset_index()
            )
            temp.columns = ["SK_ID_CURR", f"prev_{segment_name}_{recent_col.replace('prev_is_', '')}_count"]
            previous_features = previous_features.merge(temp, on="SK_ID_CURR", how="left")

previous_features.head()

,SK_ID_CURR,prev_application_count,prev_unique_contract_type_count,prev_unique_contract_status_count,prev_unique_client_type_count,prev_unique_portfolio_count,prev_unique_channel_count,prev_unique_product_combination_count,prev_approved_count,prev_approved_ratio,...,prev_unused_offer_credit_to_application_ratio_mean,prev_unused_offer_credit_to_application_ratio_max,prev_unused_offer_annuity_to_credit_ratio_sum,prev_unused_offer_annuity_to_credit_ratio_mean,prev_unused_offer_annuity_to_credit_ratio_max,prev_unused_offer_application_credit_diff_sum,prev_unused_offer_application_credit_diff_mean,prev_unused_offer_application_credit_diff_max,prev_unused_offer_recent_12m_count,prev_unused_offer_recent_24m_count
0,100001,1,1,1,1,1,1,1,1,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,100002,1,1,1,1,1,1,1,1,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100003,3,2,1,2,2,3,3,3,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,100004,1,1,1,1,1,1,1,1,1.0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,100005,2,2,2,2,2,2,2,1,0.5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [51]:
# Previous feature block basic information

previous_features = previous_features.replace([np.inf, -np.inf], np.nan)

previous_feature_block_info = pd.DataFrame([{
    "feature_block": "previous_features",
    "rows": previous_features.shape[0],
    "columns": previous_features.shape[1],
    "unique_SK_ID_CURR": previous_features["SK_ID_CURR"].nunique(),
    "duplicate_SK_ID_CURR": previous_features.duplicated("SK_ID_CURR").sum(),
    "missing_cells": previous_features.isna().sum().sum(),
}])

previous_feature_block_info.to_csv(
    OUTPUT_DIR / "previous_feature_block_info_draft.csv",
    index=False
)

previous_feature_block_info

,feature_block,rows,columns,unique_SK_ID_CURR,duplicate_SK_ID_CURR,missing_cells
0,previous_features,338857,203,338857,0,24246772


### 2.2.5 Latest Previous Application Snapshot

In this step, we add features from the most recent previous application of each applicant.

The most recent previous application is selected using the maximum `DAYS_DECISION` value because `DAYS_DECISION` is negative and values closer to zero are more recent.

This keeps recent Home Credit behavior visible even after applicant-level aggregation.


In [52]:
# Latest previous application snapshot

latest_previous = (
    pa.sort_values(["SK_ID_CURR", "DAYS_DECISION"], ascending=[True, False])
    .drop_duplicates("SK_ID_CURR")
    .reset_index(drop=True)
)

latest_prev_features = latest_previous[["SK_ID_CURR"]].copy()

latest_prev_feature_map = {
    "prev_is_approved": "prev_latest_approved_flag",
    "prev_is_refused": "prev_latest_refused_flag",
    "prev_is_canceled": "prev_latest_canceled_flag",
    "prev_is_unused_offer": "prev_latest_unused_offer_flag",
    
    "prev_is_cash_loan": "prev_latest_cash_loan_flag",
    "prev_is_consumer_loan": "prev_latest_consumer_loan_flag",
    "prev_is_revolving_loan": "prev_latest_revolving_loan_flag",
    
    "AMT_APPLICATION": "prev_latest_application_amount",
    "AMT_CREDIT": "prev_latest_credit_amount",
    "AMT_ANNUITY": "prev_latest_annuity_amount",
    "AMT_GOODS_PRICE": "prev_latest_goods_price",
    "AMT_DOWN_PAYMENT": "prev_latest_down_payment",
    
    "prev_credit_to_application_ratio": "prev_latest_credit_to_application_ratio",
    "prev_annuity_to_credit_ratio": "prev_latest_annuity_to_credit_ratio",
    "prev_down_payment_ratio": "prev_latest_down_payment_ratio",
    "prev_application_credit_diff": "prev_latest_application_credit_diff",
    
    "DAYS_DECISION": "prev_latest_days_decision",
    "CNT_PAYMENT": "prev_latest_cnt_payment",
    "HOUR_APPR_PROCESS_START": "prev_latest_hour_process",
    "SELLERPLACE_AREA": "prev_latest_sellerplace_area",
    
    "prev_last_appl_in_day": "prev_latest_last_appl_in_day",
    "prev_insured_on_approval": "prev_latest_insured_on_approval",
    "prev_has_down_payment": "prev_latest_has_down_payment",
    "prev_has_interest_rate": "prev_latest_has_interest_rate",
}

for source_col, new_col in latest_prev_feature_map.items():
    if source_col in latest_previous.columns:
        latest_prev_features[new_col] = latest_previous[source_col]

previous_features = previous_features.merge(
    latest_prev_features,
    on="SK_ID_CURR",
    how="left"
)

previous_features.head()

,SK_ID_CURR,prev_application_count,prev_unique_contract_type_count,prev_unique_contract_status_count,prev_unique_client_type_count,prev_unique_portfolio_count,prev_unique_channel_count,prev_unique_product_combination_count,prev_approved_count,prev_approved_ratio,...,prev_latest_down_payment_ratio,prev_latest_application_credit_diff,prev_latest_days_decision,prev_latest_cnt_payment,prev_latest_hour_process,prev_latest_sellerplace_area,prev_latest_last_appl_in_day,prev_latest_insured_on_approval,prev_latest_has_down_payment,prev_latest_has_interest_rate
0,100001,1,1,1,1,1,1,1,1,1.0,...,0.101468,1048.5,-1740,8.0,13,23,1,0.0,1,0
1,100002,1,1,1,1,1,1,1,1,1.0,...,0.000000,0.0,-606,24.0,9,500,1,0.0,0,0
2,100003,3,2,1,2,2,3,3,3,1.0,...,NaN,-135882.0,-746,12.0,12,-1,1,1.0,0,0
3,100004,1,1,1,1,1,1,1,1,1.0,...,0.200148,4176.0,-815,4.0,5,30,1,0.0,1,0
4,100005,2,2,2,2,2,2,2,1,0.5,...,NaN,0.0,-315,NaN,10,-1,1,NaN,0,0


### 2.2.6 Previous Feature Block Quality Check and Save

In this step, we check and save the final `previous_features` block.

The goal is to confirm that the block is applicant-level and ready for later integration.

No feature is removed in this step.


In [53]:
# Final previous feature block quality check and save

previous_features = previous_features.replace([np.inf, -np.inf], np.nan)

previous_feature_block_info = pd.DataFrame([{
    "feature_block": "previous_features",
    "rows": previous_features.shape[0],
    "columns": previous_features.shape[1],
    "unique_SK_ID_CURR": previous_features["SK_ID_CURR"].nunique(),
    "duplicate_SK_ID_CURR": previous_features.duplicated("SK_ID_CURR").sum(),
    "total_missing_cells": previous_features.isna().sum().sum(),
    "missing_columns_count": (previous_features.isna().sum() > 0).sum(),
}])

previous_feature_missing_info = (
    previous_features
    .isna()
    .mean()
    .mul(100)
    .reset_index()
)

previous_feature_missing_info.columns = ["feature_name", "missing_percent"]

previous_feature_missing_info = (
    previous_feature_missing_info
    .query("missing_percent > 0")
    .sort_values("missing_percent", ascending=False)
    .reset_index(drop=True)
)

previous_features.to_csv(OUTPUT_DIR / "previous_features.csv", index=False)
previous_feature_block_info.to_csv(OUTPUT_DIR / "previous_feature_block_info.csv", index=False)
previous_feature_missing_info.to_csv(OUTPUT_DIR / "previous_feature_missing_info.csv", index=False)

previous_feature_block_info

,feature_block,rows,columns,unique_SK_ID_CURR,duplicate_SK_ID_CURR,total_missing_cells,missing_columns_count
0,previous_features,338857,227,338857,0,25247878,130


In [54]:
previous_feature_missing_info.head(30)

,feature_name,missing_percent
0,prev_canceled_down_payment_mean,99.826771
1,prev_canceled_down_payment_max,99.826771
2,prev_unused_offer_annuity_to_credit_ratio_max,99.753288
3,prev_unused_offer_annuity_to_credit_ratio_mean,99.753288
4,prev_unused_offer_annuity_amount_max,99.753288
5,prev_unused_offer_annuity_amount_mean,99.753288
6,prev_unused_offer_cnt_payment_max,99.753288
7,prev_unused_offer_cnt_payment_mean,99.753288
8,prev_canceled_annuity_amount_mean,97.212689
9,prev_canceled_annuity_amount_max,97.212689


### 2.2 Final Decision: Previous Application Feature Block

| Area               | Observation                                                                                                                                              | Decision                                                                                                              |
| ------------------ | -------------------------------------------------------------------------------------------------------------------------------------------------------- | --------------------------------------------------------------------------------------------------------------------- |
| Relationship       | `previous_application` contains multiple previous Home Credit applications per applicant.                                                                | It was aggregated by `SK_ID_CURR` to create applicant-level features.                                                 |
| Main signal        | The table captures past Home Credit relationship, approval/refusal behavior, previous loan amount, repayment burden, product type, channel, and recency. | These features are important candidate signals for credit risk modeling.                                              |
| Row-level features | Outcome flags, contract flags, recency flags, amount ratios, amount gaps, payment-term flags, and process indicators were created before aggregation.    | These helper features make the applicant-level summaries more meaningful.                                             |
| Latest snapshot    | The most recent previous application was added separately using maximum `DAYS_DECISION`.                                                                 | This preserves recent application behavior after aggregation.                                                         |
| Final block        | `previous_features` is now one row per `SK_ID_CURR`.                                                                                                     | It is safe for later merging with the application base.                                                               |
| Feature filtering  | No feature is removed in this data-engineering step.                                                                                                     | Filtering will be handled later using missingness, correlation, Cramér’s V, feature importance, and validation score. |

**Final decision:** `previous_application` was successfully converted into an applicant-level candidate feature block named `previous_features`. This block will be saved and used later during final integration.
